# AMR Gene Family Distribution, per Species

Counts how many AMR genes fall under each AMR gene family, for each of the six ESKAPE pathogens.

Uses the per-gene-family FASTA files already extracted in `Results/Genefamily/<Org>_ARG_Fasta_Files/<gene family name>.fasta` (built in `ARG_Family_classification.ipynb`) — each FASTA record in one of these files is a single RGI ARG hit belonging to that gene family, so counting records per file gives the gene count per family per species directly, with no need to re-parse RGI/CARD_Results.

This is the count version of the existing binary presence/absence matrix (`Results/DRUG_class_family_mapping/ESKAPE_AMR_Presence_Absence_Matrix.csv`) — same six species, same gene families, but actual occurrence counts instead of 1/0.

**Note:** This notebook only defines the pipeline; no cells have been run yet.

## 1. Configuration

In [1]:
# Standard library imports used for file discovery and path handling.
import os
import glob
import pandas as pd

BASE_DIR = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family'
GENEFAMILY_DIR = os.path.join(BASE_DIR, 'Results/Genefamily')

Organism folder suffix -> full species name, same six organisms and naming convention used throughout this project's other notebooks.

In [2]:
# Organism code -> (ARG_Fasta_Files folder name, full species name).
ORGANISMS = {
    'AB':  ('AB_ARG_Fasta_Files',  'Acinetobacter baumannii'),
    'EF':  ('EF_ARG_Fasta_Files',  'Enterococcus faecium'),
    'ENT': ('ENT_ARG_Fasta_Files', 'Enterobacter spp.'),
    'Kp':  ('Kp_ARG_Fasta_Files',  'Klebsiella pneumoniae'),
    'Pa':  ('Pa_ARG_Fasta_Files',  'Pseudomonas aeruginosa'),
    'SA':  ('SA_ARG_Fasta_Files',  'Staphylococcus aureus'),
}

## 2. Shared helper

Pure file I/O (counting FASTA records) — not the analysis itself, so kept as one small reusable function rather than repeated six times.

In [3]:
def count_gene_family_fastas(org_fasta_dir):
    """
    Count FASTA records (= RGI ARG hits) per gene family for one organism.

    Each file in org_fasta_dir is named after its AMR gene family (e.g. "NDM beta-lactamase.fasta"),
    and every '>' header line inside it is one gene hit belonging to that family.
    """
    counts = {}
    for fasta_path in glob.glob(os.path.join(org_fasta_dir, '*.fasta')):
        gene_family = os.path.splitext(os.path.basename(fasta_path))[0]
        with open(fasta_path) as f:
            counts[gene_family] = sum(1 for line in f if line.startswith('>'))
    return counts

## 3. Per-organism gene family counts

One explicit cell per organism, each producing its own named variable — nothing hidden in a loop, consistent with this project's established notebooks.

In [4]:
# Acinetobacter baumannii (AB): count gene hits per AMR gene family.
AB_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'AB_ARG_Fasta_Files'))
print(f'AB: {len(AB_family_counts)} gene families, {sum(AB_family_counts.values())} total gene hits')

AB: 50 gene families, 33629 total gene hits


In [5]:
# Enterococcus faecium (EF): count gene hits per AMR gene family.
EF_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'EF_ARG_Fasta_Files'))
print(f'EF: {len(EF_family_counts)} gene families, {sum(EF_family_counts.values())} total gene hits')

EF: 32 gene families, 17744 total gene hits


In [6]:
# Enterobacter spp. (ENT): count gene hits per AMR gene family.
ENT_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'ENT_ARG_Fasta_Files'))
print(f'ENT: {len(ENT_family_counts)} gene families, {sum(ENT_family_counts.values())} total gene hits')

ENT: 73 gene families, 29168 total gene hits


In [7]:
# Klebsiella pneumoniae (Kp): count gene hits per AMR gene family.
Kp_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'Kp_ARG_Fasta_Files'))
print(f'Kp: {len(Kp_family_counts)} gene families, {sum(Kp_family_counts.values())} total gene hits')

Kp: 77 gene families, 189098 total gene hits


In [8]:
# Pseudomonas aeruginosa (Pa): count gene hits per AMR gene family.
Pa_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'Pa_ARG_Fasta_Files'))
print(f'Pa: {len(Pa_family_counts)} gene families, {sum(Pa_family_counts.values())} total gene hits')

Pa: 63 gene families, 94360 total gene hits


In [9]:
# Staphylococcus aureus (SA): count gene hits per AMR gene family.
SA_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'SA_ARG_Fasta_Files'))
print(f'SA: {len(SA_family_counts)} gene families, {sum(SA_family_counts.values())} total gene hits')

SA: 50 gene families, 52847 total gene hits


## 4. Combine into a long-format table

One row per (species, gene family) combination — the easiest shape for downstream filtering, grouping, or plotting.

In [10]:
# Combine all six organisms' count dicts into one long-format dataframe.
long_rows = []
for code_, (folder, species) in ORGANISMS.items():
    counts = {'AB': AB_family_counts, 'EF': EF_family_counts, 'ENT': ENT_family_counts,
              'Kp': Kp_family_counts, 'Pa': Pa_family_counts, 'SA': SA_family_counts}[code_]
    for gene_family, count in counts.items():
        long_rows.append({'Organism': species, 'AMR Gene Family': gene_family, 'Count': count})

gene_family_counts_long_df = pd.DataFrame(long_rows)
print(f'{len(gene_family_counts_long_df)} (species, gene family) rows total')
gene_family_counts_long_df.head()

345 (species, gene family) rows total


,Organism,AMR Gene Family,Count
0,Acinetobacter baumannii,APH(4),2
1,Acinetobacter baumannii,resistance-nodulation-cell division (RND) anti...,12138
2,Acinetobacter baumannii,fluoroquinolone resistant parC,965
3,Acinetobacter baumannii,small multidrug resistance (SMR) antibiotic ef...,978
4,Acinetobacter baumannii,GIM beta-lactamase,1


## 5. Pivot into a wide matrix

Species x gene family, count values, 0 where a species doesn't have that gene family — mirrors the existing `ESKAPE_AMR_Presence_Absence_Matrix.csv` structure exactly, but with counts instead of 1/0.

In [11]:
# Pivot to species (rows) x gene family (columns), filling absent combinations with 0.
gene_family_counts_matrix_df = (
    gene_family_counts_long_df
    .pivot(index='Organism', columns='AMR Gene Family', values='Count')
    .fillna(0)
    .astype(int)
    .reindex(sorted(gene_family_counts_long_df['AMR Gene Family'].unique()), axis=1)
)
gene_family_counts_matrix_df

AMR Gene Family,16S rRNA methyltransferase (G1405),23S rRNA with mutation conferring resistance to linezolid antibiotics,AAC(2'),AAC(3),AAC(6'),AAC(6')-Ib-cr,ACC beta-lactamase,ACT beta-lactamase,ADC beta-lactamase without carbapenemase activity,ADC beta-lactamases pending classification for carbapenemase activity,...,undecaprenyl pyrophosphate related proteins,vanH,vanR,vanS,vanT,vanW,vanX,vanY,vanZ,vga-type ABC-F protein
Organism,,,,,,,,,,,,,,,,,,,,,
Acinetobacter baumannii,425,0,0,225,280,0,0,0,557,442,...,0,0,0,0,0,0,0,0,0,0
Enterobacter spp.,35,0,0,159,336,0,11,548,0,0,...,611,0,0,0,0,0,0,0,0,0
Enterococcus faecium,0,0,0,0,762,0,0,0,0,0,...,0,708,705,419,0,53,725,1498,495,0
Klebsiella pneumoniae,908,0,1,1287,2169,6,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Pseudomonas aeruginosa,66,0,1,25,93,0,0,0,0,0,...,0,1191,0,0,250,1367,0,0,0,0
Staphylococcus aureus,0,6,2,0,0,0,0,0,0,0,...,0,12,12,11,2236,1,12,56,12,10


In [12]:
os.makedirs('Results/AMR_GeneDistribution', exist_ok=True)

## 6. Save

Both the long-format table and the wide matrix, alongside the project's other `Results/` outputs.

In [ ]:
'''
# Save both output shapes.
gene_family_counts_long_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Counts_long.csv'), index=False
)
gene_family_counts_matrix_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Counts_matrix.csv')
)
'''

## 7. Correction: count unique genes per family, not raw hits

Nothing above this point was modified. The counting method above (`count_gene_family_fastas`, cell 6) counts every FASTA record — every RGI hit — which double-counts a gene family when the same genome carries more than one copy of the same gene. Verified directly: 2 of AB's 967 genomes carry `CARB-16` more than once (`GCF_029713805.1` twice, `GCF_050032045.1` three times), so AB's `CARB beta-lactamase.fasta` has 44 raw records but only 3 **unique gene names** (`CARB-16`, `CARB-2`, `CARB-50`).

This section redefines the counting function to count **unique gene names** (distinct FASTA headers) per family instead, recomputes all six organisms with it, and adds each family's count as a percentage of that organism's total number of gene families.

In [13]:
def count_gene_family_fastas(org_fasta_dir):
    """
    """
    counts = {}
    for fasta_path in glob.glob(os.path.join(org_fasta_dir, '*.fasta')):
        gene_family = os.path.splitext(os.path.basename(fasta_path))[0]
        with open(fasta_path) as f:
            headers = {line.strip() for line in f if line.startswith('>')}
        counts[gene_family] = len(headers)
    return counts

In [14]:
# Acinetobacter baumannii (AB): recompute with the corrected (unique-gene) counting function.
AB_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'AB_ARG_Fasta_Files'))
print(f'AB: {len(AB_family_counts)} gene families, {sum(AB_family_counts.values())} total unique genes')

AB: 50 gene families, 371 total unique genes


In [15]:
# Enterococcus faecium (EF): recompute with the corrected (unique-gene) counting function.
EF_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'EF_ARG_Fasta_Files'))
print(f'EF: {len(EF_family_counts)} gene families, {sum(EF_family_counts.values())} total unique genes')

EF: 32 gene families, 93 total unique genes


In [16]:
# Enterobacter spp. (ENT): recompute with the corrected (unique-gene) counting function.
ENT_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'ENT_ARG_Fasta_Files'))
print(f'ENT: {len(ENT_family_counts)} gene families, {sum(ENT_family_counts.values())} total unique genes')

ENT: 73 gene families, 359 total unique genes


In [17]:
# Klebsiella pneumoniae (Kp): recompute with the corrected (unique-gene) counting function.
Kp_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'Kp_ARG_Fasta_Files'))
print(f'Kp: {len(Kp_family_counts)} gene families, {sum(Kp_family_counts.values())} total unique genes')

Kp: 77 gene families, 477 total unique genes


In [18]:
# Pseudomonas aeruginosa (Pa): recompute with the corrected (unique-gene) counting function.
Pa_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'Pa_ARG_Fasta_Files'))
print(f'Pa: {len(Pa_family_counts)} gene families, {sum(Pa_family_counts.values())} total unique genes')

Pa: 63 gene families, 508 total unique genes


In [19]:
# Staphylococcus aureus (SA): recompute with the corrected (unique-gene) counting function.
SA_family_counts = count_gene_family_fastas(os.path.join(GENEFAMILY_DIR, 'SA_ARG_Fasta_Files'))
print(f'SA: {len(SA_family_counts)} gene families, {sum(SA_family_counts.values())} total unique genes')

SA: 50 gene families, 107 total unique genes


In [20]:
# Rebuild the long-format table with the corrected (unique-gene) counts.
long_rows = []
for code_, (folder, species) in ORGANISMS.items():
    counts = {'AB': AB_family_counts, 'EF': EF_family_counts, 'ENT': ENT_family_counts,
              'Kp': Kp_family_counts, 'Pa': Pa_family_counts, 'SA': SA_family_counts}[code_]
    for gene_family, count in counts.items():
        long_rows.append({'Organism': species, 'AMR Gene Family': gene_family, 'Count': count})

gene_family_counts_long_df = pd.DataFrame(long_rows)
print(f'{len(gene_family_counts_long_df)} (species, gene family) rows total')
gene_family_counts_long_df.head()

345 (species, gene family) rows total


,Organism,AMR Gene Family,Count
0,Acinetobacter baumannii,APH(4),1
1,Acinetobacter baumannii,resistance-nodulation-cell division (RND) anti...,14
2,Acinetobacter baumannii,fluoroquinolone resistant parC,1
3,Acinetobacter baumannii,small multidrug resistance (SMR) antibiotic ef...,3
4,Acinetobacter baumannii,GIM beta-lactamase,1


In [21]:
# Add each family's count as a percentage of that organism's own total number of gene
# families (each organism's total_families is its own denominator, not shared across all six).
total_families_by_species = {
    species: len(counts) for species, counts in
    zip(
        [s for _, s in ORGANISMS.values()],
        [AB_family_counts, EF_family_counts, ENT_family_counts, Kp_family_counts, Pa_family_counts, SA_family_counts],
    )
}
gene_family_counts_long_df['Percent of Total Families'] = gene_family_counts_long_df.apply(
    lambda row: row['Count'] / total_families_by_species[row['Organism']] * 100,
    axis=1,
)
gene_family_counts_long_df.head()

,Organism,AMR Gene Family,Count,Percent of Total Families
0,Acinetobacter baumannii,APH(4),1,2.0
1,Acinetobacter baumannii,resistance-nodulation-cell division (RND) anti...,14,28.0
2,Acinetobacter baumannii,fluoroquinolone resistant parC,1,2.0
3,Acinetobacter baumannii,small multidrug resistance (SMR) antibiotic ef...,3,6.0
4,Acinetobacter baumannii,GIM beta-lactamase,1,2.0


In [22]:
# Rebuild the wide matrix (species x gene family) from the corrected counts.
gene_family_counts_matrix_df = (
    gene_family_counts_long_df
    .pivot(index='Organism', columns='AMR Gene Family', values='Count')
    .fillna(0)
    .astype(int)
    .reindex(sorted(gene_family_counts_long_df['AMR Gene Family'].unique()), axis=1)
)
gene_family_counts_matrix_df

AMR Gene Family,16S rRNA methyltransferase (G1405),23S rRNA with mutation conferring resistance to linezolid antibiotics,AAC(2'),AAC(3),AAC(6'),AAC(6')-Ib-cr,ACC beta-lactamase,ACT beta-lactamase,ADC beta-lactamase without carbapenemase activity,ADC beta-lactamases pending classification for carbapenemase activity,...,undecaprenyl pyrophosphate related proteins,vanH,vanR,vanS,vanT,vanW,vanX,vanY,vanZ,vga-type ABC-F protein
Organism,,,,,,,,,,,,,,,,,,,,,
Acinetobacter baumannii,2,0,0,5,4,0,0,0,19,77,...,0,0,0,0,0,0,0,0,0,0
Enterobacter spp.,3,0,0,5,13,0,1,85,0,0,...,1,0,0,0,0,0,0,0,0,0
Enterococcus faecium,0,0,0,0,1,0,0,0,0,0,...,0,4,4,3,0,1,4,4,1,0
Klebsiella pneumoniae,6,0,1,7,18,2,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Pseudomonas aeruginosa,4,0,1,3,8,0,0,0,0,0,...,0,2,0,0,1,1,0,0,0,0
Staphylococcus aureus,0,1,1,0,0,0,0,0,0,0,...,0,1,1,1,1,1,1,2,1,1


In [ ]:
'''
# Re-save both output shapes with the corrected counts, to the same location already in use.
gene_family_counts_long_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Counts_percentage.csv'), index=False
)
gene_family_counts_matrix_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Counts_matrix_percent.csv')
)
'''

In [23]:
# Pivot the long-form percentage values (not counts) into a species x gene family matrix.
# gene_family_counts_matrix_df (above) is pivoted on 'Count' -- this one uses
# 'Percent of Total Families' instead, so cell values are the actual percentages.
gene_family_percent_matrix_df = (
    gene_family_counts_long_df
    .pivot(index='Organism', columns='AMR Gene Family', values='Percent of Total Families')
    .fillna(0)
    .reindex(sorted(gene_family_counts_long_df['AMR Gene Family'].unique()), axis=1)
)
gene_family_percent_matrix_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Percentage_Matrix.csv')
)
gene_family_percent_matrix_df

AMR Gene Family,16S rRNA methyltransferase (G1405),23S rRNA with mutation conferring resistance to linezolid antibiotics,AAC(2'),AAC(3),AAC(6'),AAC(6')-Ib-cr,ACC beta-lactamase,ACT beta-lactamase,ADC beta-lactamase without carbapenemase activity,ADC beta-lactamases pending classification for carbapenemase activity,...,undecaprenyl pyrophosphate related proteins,vanH,vanR,vanS,vanT,vanW,vanX,vanY,vanZ,vga-type ABC-F protein
Organism,,,,,,,,,,,,,,,,,,,,,
Acinetobacter baumannii,4.000000,0.0,0.000000,10.000000,8.000000,0.000000,0.000000,0.000000,38.0,154.0,...,0.000000,0.000000,0.0,0.000,0.000000,0.000000,0.0,0.0,0.000,0.0
Enterobacter spp.,4.109589,0.0,0.000000,6.849315,17.808219,0.000000,1.369863,116.438356,0.0,0.0,...,1.369863,0.000000,0.0,0.000,0.000000,0.000000,0.0,0.0,0.000,0.0
Enterococcus faecium,0.000000,0.0,0.000000,0.000000,3.125000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,12.500000,12.5,9.375,0.000000,3.125000,12.5,12.5,3.125,0.0
Klebsiella pneumoniae,7.792208,0.0,1.298701,9.090909,23.376623,2.597403,1.298701,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.000,0.000000,0.000000,0.0,0.0,0.000,0.0
Pseudomonas aeruginosa,6.349206,0.0,1.587302,4.761905,12.698413,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,3.174603,0.0,0.000,1.587302,1.587302,0.0,0.0,0.000,0.0
Staphylococcus aureus,0.000000,2.0,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,2.000000,2.0,2.000,2.000000,2.000000,2.0,4.0,2.000,2.0


## 8. AMR gene families shared across all six ESKAPE pathogens

`gene_family_counts_long_df` only has a row for an (organism, gene family) pair when that family is actually present in that organism (built from each organism's own `ARG_Fasta_Files` folder). So a family shared by all six pathogens is one whose number of distinct organisms in this table equals the total number of organisms — computed from the data itself, not hardcoded to 6, so this stays correct if organism coverage changes later.

In [24]:
# How many distinct organisms is each gene family present in?
family_organism_counts = gene_family_counts_long_df.groupby('AMR Gene Family')['Organism'].nunique()
n_total_organisms = gene_family_counts_long_df['Organism'].nunique()

shared_families = sorted(family_organism_counts[family_organism_counts == n_total_organisms].index)
print(f'{len(shared_families)} of {len(family_organism_counts)} gene families are present in all {n_total_organisms} ESKAPE pathogens:')
for fam in shared_families:
    print(' ', fam)

7 of 142 gene families are present in all 6 ESKAPE pathogens:
  APH(3')
  Erm 23S ribosomal RNA methyltransferase
  chloramphenicol acetyltransferase (CAT)
  fosfomycin thiol transferase
  major facilitator superfamily (MFS) antibiotic efflux pump
  msr-type ABC-F protein
  small multidrug resistance (SMR) antibiotic efflux pump


In [25]:
# Side-by-side view: each shared family's Count and Percent of Total Families, per organism.
shared_families_detail_df = (
    gene_family_counts_long_df[gene_family_counts_long_df['AMR Gene Family'].isin(shared_families)]
    .pivot(index='AMR Gene Family', columns='Organism', values=['Count', 'Percent of Total Families'])
    .reindex(shared_families)
)
shared_families_detail_df

Count  \
Organism                                           Acinetobacter baumannii   
AMR Gene Family                                                              
APH(3')                                                                3.0   
Erm 23S ribosomal RNA methyltransferase                                1.0   
chloramphenicol acetyltransferase (CAT)                                5.0   
fosfomycin thiol transferase                                           2.0   
major facilitator superfamily (MFS) antibiotic ...                    16.0   
msr-type ABC-F protein                                                 1.0   
small multidrug resistance (SMR) antibiotic eff...                     3.0   

                                                                      \
Organism                                           Enterobacter spp.   
AMR Gene Family                                                        
APH(3')                                                          4.0   
Erm 23S ribosomal RNA methyltransferase                          1.0   
chloramphenicol acetyltransferase (CAT)                          6.0   
fosfomycin thiol transferase                                     8.0   
major facilitator superfamily (MFS) antibiotic ...              20.0   
msr-type ABC-F protein                                           1.0   
small multidrug resistance (SMR) antibiotic eff...               5.0   

                                                                         \
Organism                                           Enterococcus faecium   
AMR Gene Family                                                           
APH(3')                                                             1.0   
Erm 23S ribosomal RNA methyltransferase                             4.0   
chloramphenicol acetyltransferase (CAT)                             5.0   
fosfomycin thiol transferase                                        2.0   
major facilitator superfamily (MFS) antibiotic ...                  6.0   
msr-type ABC-F protein                                              1.0   
small multidrug resistance (SMR) antibiotic eff...                  1.0   

                                                                          \
Organism                                           Klebsiella pneumoniae   
AMR Gene Family                                                            
APH(3')                                                              6.0   
Erm 23S ribosomal RNA methyltransferase                              2.0   
chloramphenicol acetyltransferase (CAT)                              6.0   
fosfomycin thiol transferase                                         7.0   
major facilitator superfamily (MFS) antibiotic ...                  23.0   
msr-type ABC-F protein                                               1.0   
small multidrug resistance (SMR) antibiotic eff...                   5.0   

                                                                           \
Organism                                           Pseudomonas aeruginosa   
AMR Gene Family                                                             
APH(3')                                                               6.0   
Erm 23S ribosomal RNA methyltransferase                               2.0   
chloramphenicol acetyltransferase (CAT)                               6.0   
fosfomycin thiol transferase                                          2.0   
major facilitator superfamily (MFS) antibiotic ...                   15.0   
msr-type ABC-F protein                                                1.0   
small multidrug resistance (SMR) antibiotic eff...                    2.0   

                                                                          \
Organism                                           Staphylococcus aureus   
AMR Gene Family                                                            
APH(3')                                                 

In [ ]:
'''
# Save the shared-family list with per-organism detail.
shared_families_detail_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_Shared_AMR_Gene_Families.csv')
)
'''

## 9. Heatmap: each gene family's count across all six ESKAPE pathogens

Rows = gene family, columns = pathogen, cell = gene count for that family in that pathogen -- so any one family's row shows its count in every pathogen side by side (e.g. `fluoroquinolone resistant parC`: 1 in AB, 0 in EF, 1 in ENT, 1 in Kp, 0 in Pa, 1 in SA). All 142 families are shown, sorted by total count across all six pathogens (descending), so the most broadly/heavily present families sit at the top.

Uses `ESKAPE_AMR_Gene_Family_Counts_matrix.csv` (already built, species x gene family, raw counts). Colormap is a single sequential hue (blue, light->dark) per the `dataviz` skill -- magnitude gets one hue, never a rainbow.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load the species x gene family count matrix (correct filename -- see note above).
count_matrix_df = pd.read_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Counts_matrix_percent.csv'),
    index_col=0,
)

# ESKAPE acronym order, matching the earlier chart.
eskape_order = [
    'Enterococcus faecium', 'Staphylococcus aureus', 'Klebsiella pneumoniae',
    'Acinetobacter baumannii', 'Pseudomonas aeruginosa', 'Enterobacter spp.',
]
count_matrix_df = count_matrix_df.reindex(eskape_order)

# Transpose so rows = gene family, columns = pathogen, then sort rows by total count, descending.
family_by_pathogen_df = count_matrix_df.T
family_by_pathogen_df['__total__'] = family_by_pathogen_df.sum(axis=1)
family_by_pathogen_df = family_by_pathogen_df.sort_values('__total__', ascending=False).drop(columns='__total__')

fig, ax = plt.subplots(figsize=(6, 0.16 * len(family_by_pathogen_df) + 1.5))

im = ax.imshow(family_by_pathogen_df.values, aspect='auto', cmap='Blues')

ax.set_xticks(range(len(eskape_order)))
ax.set_xticklabels(eskape_order, rotation=30, ha='right', color='#0b0b0b')
ax.set_yticks(range(len(family_by_pathogen_df)))
ax.set_yticklabels(family_by_pathogen_df.index, fontsize=6, color='#0b0b0b')

ax.set_title('AMR gene family count, per ESKAPE pathogen', color='#0b0b0b')
for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(im, ax=ax, shrink=0.3, pad=0.02)
cbar.set_label('Gene count', color='#0b0b0b')

plt.tight_layout()
'''
#plt.savefig(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_Gene_Family_Heatmap.png'),
    dpi=200,
)
'''
plt.show()

## 10. Correction: percentage of total genes, not total families

Nothing above this point was modified. The `Percent of Total Families` column (cell 31) divides each family's `Count` by the *number of gene families* for that organism (e.g. 50 for AB) -- that's not a share-of-whole metric, so it doesn't sum to 100% and can exceed 100% for large families. Verified: AB's `OXA beta-lactamase` alone has 80 unique genes, and 80/50*100 = 160%; summed across all of AB's 142 family rows, the column totals 742%, not 100%.

This section adds a proper share-of-whole percentage: `Count / total_genes_for_that_organism * 100`, where `total_genes_for_that_organism` is the sum of `Count` across all of that organism's own families (371 for AB) -- bounded 0-100, and sums to 100% within each organism.

In [26]:
# Add a true "percent of total genes" column: Count / total unique genes for that
# organism * 100 -- each organism's own gene total is its own denominator.
total_genes_by_species = gene_family_counts_long_df.groupby('Organism')['Count'].transform('sum')
gene_family_counts_long_df['Percent of Total Genes'] = (
    gene_family_counts_long_df['Count'] / total_genes_by_species * 100
)

# Sanity check: each organism's column should now sum to 100.
print(gene_family_counts_long_df.groupby('Organism')['Percent of Total Genes'].sum())
gene_family_counts_long_df.head()

Organism
Acinetobacter baumannii    100.0
Enterobacter spp.          100.0
Enterococcus faecium       100.0
Klebsiella pneumoniae      100.0
Pseudomonas aeruginosa     100.0
Staphylococcus aureus      100.0
Name: Percent of Total Genes, dtype: float64


,Organism,AMR Gene Family,Count,Percent of Total Families,Percent of Total Genes
0,Acinetobacter baumannii,APH(4),1,2.0,0.269542
1,Acinetobacter baumannii,resistance-nodulation-cell division (RND) anti...,14,28.0,3.773585
2,Acinetobacter baumannii,fluoroquinolone resistant parC,1,2.0,0.269542
3,Acinetobacter baumannii,small multidrug resistance (SMR) antibiotic ef...,3,6.0,0.808625
4,Acinetobacter baumannii,GIM beta-lactamase,1,2.0,0.269542


In [27]:
# Pivot the new percent-of-genes values into a species x gene family matrix, and save both.
gene_family_percent_of_genes_matrix_df = (
    gene_family_counts_long_df
    .pivot(index='Organism', columns='AMR Gene Family', values='Percent of Total Genes')
    .fillna(0)
    .reindex(sorted(gene_family_counts_long_df['AMR Gene Family'].unique()), axis=1)
)

gene_family_counts_long_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Counts_percentage_2.csv'), index=False
)
gene_family_percent_of_genes_matrix_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Percent_of_Genes_Matrix_3.csv')
)
gene_family_percent_of_genes_matrix_df

AMR Gene Family,16S rRNA methyltransferase (G1405),23S rRNA with mutation conferring resistance to linezolid antibiotics,AAC(2'),AAC(3),AAC(6'),AAC(6')-Ib-cr,ACC beta-lactamase,ACT beta-lactamase,ADC beta-lactamase without carbapenemase activity,ADC beta-lactamases pending classification for carbapenemase activity,...,undecaprenyl pyrophosphate related proteins,vanH,vanR,vanS,vanT,vanW,vanX,vanY,vanZ,vga-type ABC-F protein
Organism,,,,,,,,,,,,,,,,,,,,,
Acinetobacter baumannii,0.539084,0.000000,0.000000,1.347709,1.078167,0.000000,0.000000,0.00000,5.121294,20.754717,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Enterobacter spp.,0.835655,0.000000,0.000000,1.392758,3.621170,0.000000,0.278552,23.67688,0.000000,0.000000,...,0.278552,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Enterococcus faecium,0.000000,0.000000,0.000000,0.000000,1.075269,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.000000,4.301075,4.301075,3.225806,0.000000,1.075269,4.301075,4.301075,1.075269,0.000000
Klebsiella pneumoniae,1.257862,0.000000,0.209644,1.467505,3.773585,0.419287,0.209644,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Pseudomonas aeruginosa,0.787402,0.000000,0.196850,0.590551,1.574803,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.393701,0.000000,0.000000,0.196850,0.196850,0.000000,0.000000,0.000000,0.000000
Staphylococcus aureus,0.000000,0.934579,0.934579,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.934579,0.934579,0.934579,0.934579,0.934579,0.934579,1.869159,0.934579,0.934579


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load the species x gene family count matrix (correct filename -- see note above).
count_matrix_df = pd.read_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Percent_of_Genes_Matrix_3.csv'),
    index_col=0,
)

# ESKAPE acronym order, matching the earlier chart.
eskape_order = [
    'Enterococcus faecium', 'Staphylococcus aureus', 'Klebsiella pneumoniae',
    'Acinetobacter baumannii', 'Pseudomonas aeruginosa', 'Enterobacter spp.',
]
count_matrix_df = count_matrix_df.reindex(eskape_order)

# Transpose so rows = gene family, columns = pathogen, then sort rows by total count, descending.
family_by_pathogen_df = count_matrix_df.T
family_by_pathogen_df['__total__'] = family_by_pathogen_df.sum(axis=1)
family_by_pathogen_df = family_by_pathogen_df.sort_values('__total__', ascending=False).drop(columns='__total__')

fig, ax = plt.subplots(figsize=(6, 0.16 * len(family_by_pathogen_df) + 1.5))

im = ax.imshow(family_by_pathogen_df.values, aspect='auto', cmap='Blues')

ax.set_xticks(range(len(eskape_order)))
ax.set_xticklabels(eskape_order, rotation=30, ha='right', color='#0b0b0b')
ax.set_yticks(range(len(family_by_pathogen_df)))
ax.set_yticklabels(family_by_pathogen_df.index, fontsize=6, color='#0b0b0b')

ax.set_title('AMR gene family count, per ESKAPE pathogen', color='#0b0b0b')
for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(im, ax=ax, shrink=0.3, pad=0.02)
cbar.set_label('Gene count', color='#0b0b0b')

plt.tight_layout()
'''
plt.savefig(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_Gene_Family_Heatmap_2.png'),
    dpi=200,
)
'''
plt.show()

## 11. Shared AMR gene families, gram-negative ESKAPE pathogens only

Same logic as section 8, restricted to the four gram-negative organisms (Klebsiella pneumoniae, Acinetobacter baumannii, Pseudomonas aeruginosa, Enterobacter spp.), excluding the two gram-positive organisms (Enterococcus faecium, Staphylococcus aureus).

### Find the common gene across Gram Negative Bacteria

In [28]:
# Restrict to the four gram-negative ESKAPE pathogens (excludes Enterococcus faecium
# and Staphylococcus aureus, the panel's two gram-positive organisms).
gram_negative_organisms = [
    'Klebsiella pneumoniae', 'Acinetobacter baumannii',
    'Pseudomonas aeruginosa', 'Enterobacter spp.',
]
gram_negative_long_df = gene_family_counts_long_df[
    gene_family_counts_long_df['Organism'].isin(gram_negative_organisms)
]

# How many distinct organisms is each gene family present in? (now out of 4, not 6)
family_organism_counts = gram_negative_long_df.groupby('AMR Gene Family')['Organism'].nunique()
n_total_organisms = gram_negative_long_df['Organism'].nunique()

shared_families = sorted(family_organism_counts[family_organism_counts == n_total_organisms].index)
print(f'{len(shared_families)} of {len(family_organism_counts)} gene families are present in all {n_total_organisms} gram-negative ESKAPE pathogens:')
for fam in shared_families:
    print(' ', fam)

30 of 116 gene families are present in all 4 gram-negative ESKAPE pathogens:
  16S rRNA methyltransferase (G1405)
  AAC(3)
  AAC(6')
  ANT(2'')
  ANT(3'')
  APH(3'')
  APH(3')
  APH(6)
  CARB beta-lactamase
  CMY beta-lactamase
  CTX-M beta-lactamase
  Erm 23S ribosomal RNA methyltransferase
  GES beta-lactamase
  IMP beta-lactamase
  MCR phosphoethanolamine transferase
  NDM beta-lactamase
  OXA beta-lactamase
  OXA-2-like beta-lactamase
  TEM beta-lactamase
  chloramphenicol acetyltransferase (CAT)
  fluoroquinolone resistant gyrA
  fosfomycin thiol transferase
  macrolide esterase
  macrolide phosphotransferase (MPH)
  major facilitator superfamily (MFS) antibiotic efflux pump
  msr-type ABC-F protein
  quinolone resistance protein (qnr)
  resistance-nodulation-cell division (RND) antibiotic efflux pump
  small multidrug resistance (SMR) antibiotic efflux pump
  sulfonamide resistant sul


In [29]:
# Side-by-side view: each shared family's Count and Percent of Total Families, per organism
# -- same as before, just built from gram_negative_long_df instead of the full 6-organism table.
shared_families_detail_df = (
    gram_negative_long_df[gram_negative_long_df['AMR Gene Family'].isin(shared_families)]
    .pivot(index='AMR Gene Family', columns='Organism', values=['Count', 'Percent of Total Families'])
    .reindex(shared_families)
)
shared_families_detail_df

Count  \
Organism                                           Acinetobacter baumannii   
AMR Gene Family                                                              
16S rRNA methyltransferase (G1405)                                     2.0   
AAC(3)                                                                 5.0   
AAC(6')                                                                4.0   
ANT(2'')                                                               1.0   
ANT(3'')                                                               8.0   
APH(3'')                                                               1.0   
APH(3')                                                                3.0   
APH(6)                                                                 1.0   
CARB beta-lactamase                                                    3.0   
CMY beta-lactamase                                                     1.0   
CTX-M beta-lactamase                                                   3.0   
Erm 23S ribosomal RNA methyltransferase                                1.0   
GES beta-lactamase                                                     3.0   
IMP beta-lactamase                                                     1.0   
MCR phosphoethanolamine transferase                                    1.0   
NDM beta-lactamase                                                     2.0   
OXA beta-lactamase                                                    80.0   
OXA-2-like beta-lactamase                                              1.0   
TEM beta-lactamase                                                     5.0   
chloramphenicol acetyltransferase (CAT)                                5.0   
fluoroquinolone resistant gyrA                                         1.0   
fosfomycin thiol transferase                                           2.0   
macrolide esterase                                                     1.0   
macrolide phosphotransferase (MPH)                                     1.0   
major facilitator superfamily (MFS) antibiotic ...                    16.0   
msr-type ABC-F protein                                                 1.0   
quinolone resistance protein (qnr)                                     1.0   
resistance-nodulation-cell division (RND) antib...                    14.0   
small multidrug resistance (SMR) antibiotic eff...                     3.0   
sulfonamide resistant sul                                              2.0   

                                                                      \
Organism                                           Enterobacter spp.   
AMR Gene Family                                                        
16S rRNA methyltransferase (G1405)                               3.0   
AAC(3)                                                           5.0   
AAC(6')                                                         13.0   
ANT(2'')                                                         1.0   
ANT(3'')                                                         8.0   
APH(3'')                                                         1.0   
APH(3')                                                          4.0   
APH(6)                                                           1.0   
CARB beta-lactamase                                              1.0   
CMY beta-lactamase                                               2.0   
CTX-M beta-lactamase                                             8.0   
Erm 23S ribosomal RNA methyltransferase                          1.0   
GES beta-lactamase                                               1.0   
IMP beta-lactamase                                               8.0   
MCR phosphoethanolamine transferase                              5.0   
NDM beta-lactamase                                               4.0   
OXA beta-lactamase                                               8.0   
OXA-2-like beta-lactamase                             

In [ ]:
'''
# Save the shared-family list with per-organism detail.
shared_families_detail_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_Shared_AMR_Gene_Families.csv')
)
'''

## 12. Correction: shared-family detail tables should use "Percent of Total Genes"

Cell 39 (all six pathogens) and cell 53 (gram-negative only) both pivot on `Percent of Total Families` -- the flawed metric from before section 10's fix (can exceed 100%, doesn't sum to 100). Section 10 already added the correct `Percent of Total Genes` column, but these two detail tables were never updated to use it. Rebuilt here with distinct variable names (`shared_families_all_six` / `shared_families_gram_negative`), recomputed fresh rather than reusing the ambient `shared_families` name -- cells 38 and 52 both assign to that same name, so its value depends on run order and isn't safe to rely on.

In [30]:
# All six pathogens: recompute the shared-family list explicitly (don't rely on the
# ambient `shared_families` name, which cell 52 below also overwrites for its own subset).
family_organism_counts_all_six = gene_family_counts_long_df.groupby('AMR Gene Family')['Organism'].nunique()
n_total_organisms_all_six = gene_family_counts_long_df['Organism'].nunique()
shared_families_all_six = sorted(
    family_organism_counts_all_six[family_organism_counts_all_six == n_total_organisms_all_six].index
)

# Corrected detail table: Percent of Total Genes instead of Percent of Total Families.
shared_families_detail_df = (
    gene_family_counts_long_df[gene_family_counts_long_df['AMR Gene Family'].isin(shared_families_all_six)]
    .pivot(index='AMR Gene Family', columns='Organism', values=['Count', 'Percent of Total Genes'])
    .reindex(shared_families_all_six)
)
shared_families_detail_df

Count  \
Organism                                           Acinetobacter baumannii   
AMR Gene Family                                                              
APH(3')                                                                3.0   
Erm 23S ribosomal RNA methyltransferase                                1.0   
chloramphenicol acetyltransferase (CAT)                                5.0   
fosfomycin thiol transferase                                           2.0   
major facilitator superfamily (MFS) antibiotic ...                    16.0   
msr-type ABC-F protein                                                 1.0   
small multidrug resistance (SMR) antibiotic eff...                     3.0   

                                                                      \
Organism                                           Enterobacter spp.   
AMR Gene Family                                                        
APH(3')                                                          4.0   
Erm 23S ribosomal RNA methyltransferase                          1.0   
chloramphenicol acetyltransferase (CAT)                          6.0   
fosfomycin thiol transferase                                     8.0   
major facilitator superfamily (MFS) antibiotic ...              20.0   
msr-type ABC-F protein                                           1.0   
small multidrug resistance (SMR) antibiotic eff...               5.0   

                                                                         \
Organism                                           Enterococcus faecium   
AMR Gene Family                                                           
APH(3')                                                             1.0   
Erm 23S ribosomal RNA methyltransferase                             4.0   
chloramphenicol acetyltransferase (CAT)                             5.0   
fosfomycin thiol transferase                                        2.0   
major facilitator superfamily (MFS) antibiotic ...                  6.0   
msr-type ABC-F protein                                              1.0   
small multidrug resistance (SMR) antibiotic eff...                  1.0   

                                                                          \
Organism                                           Klebsiella pneumoniae   
AMR Gene Family                                                            
APH(3')                                                              6.0   
Erm 23S ribosomal RNA methyltransferase                              2.0   
chloramphenicol acetyltransferase (CAT)                              6.0   
fosfomycin thiol transferase                                         7.0   
major facilitator superfamily (MFS) antibiotic ...                  23.0   
msr-type ABC-F protein                                               1.0   
small multidrug resistance (SMR) antibiotic eff...                   5.0   

                                                                           \
Organism                                           Pseudomonas aeruginosa   
AMR Gene Family                                                             
APH(3')                                                               6.0   
Erm 23S ribosomal RNA methyltransferase                               2.0   
chloramphenicol acetyltransferase (CAT)                               6.0   
fosfomycin thiol transferase                                          2.0   
major facilitator superfamily (MFS) antibiotic ...                   15.0   
msr-type ABC-F protein                                                1.0   
small multidrug resistance (SMR) antibiotic eff...                    2.0   

                                                                          \
Organism                                           Staphylococcus aureus   
AMR Gene Family                                                            
APH(3')                                                 

In [ ]:
'''
# Save the shared-family list with per-organism detail.
shared_families_detail_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_Shared_AMR_Gene_Families.csv')
)
'''

In [31]:
# Gram-negative only: same correction, recomputed explicitly from gram_negative_long_df.
family_organism_counts_gram_negative = gram_negative_long_df.groupby('AMR Gene Family')['Organism'].nunique()
n_total_organisms_gram_negative = gram_negative_long_df['Organism'].nunique()
shared_families_gram_negative = sorted(
    family_organism_counts_gram_negative[
        family_organism_counts_gram_negative == n_total_organisms_gram_negative
    ].index
)

shared_families_detail_gram_negative_df = (
    gram_negative_long_df[gram_negative_long_df['AMR Gene Family'].isin(shared_families_gram_negative)]
    .pivot(index='AMR Gene Family', columns='Organism', values=['Count', 'Percent of Total Genes'])
    .reindex(shared_families_gram_negative)
)
shared_families_detail_gram_negative_df

Count  \
Organism                                           Acinetobacter baumannii   
AMR Gene Family                                                              
16S rRNA methyltransferase (G1405)                                     2.0   
AAC(3)                                                                 5.0   
AAC(6')                                                                4.0   
ANT(2'')                                                               1.0   
ANT(3'')                                                               8.0   
APH(3'')                                                               1.0   
APH(3')                                                                3.0   
APH(6)                                                                 1.0   
CARB beta-lactamase                                                    3.0   
CMY beta-lactamase                                                     1.0   
CTX-M beta-lactamase                                                   3.0   
Erm 23S ribosomal RNA methyltransferase                                1.0   
GES beta-lactamase                                                     3.0   
IMP beta-lactamase                                                     1.0   
MCR phosphoethanolamine transferase                                    1.0   
NDM beta-lactamase                                                     2.0   
OXA beta-lactamase                                                    80.0   
OXA-2-like beta-lactamase                                              1.0   
TEM beta-lactamase                                                     5.0   
chloramphenicol acetyltransferase (CAT)                                5.0   
fluoroquinolone resistant gyrA                                         1.0   
fosfomycin thiol transferase                                           2.0   
macrolide esterase                                                     1.0   
macrolide phosphotransferase (MPH)                                     1.0   
major facilitator superfamily (MFS) antibiotic ...                    16.0   
msr-type ABC-F protein                                                 1.0   
quinolone resistance protein (qnr)                                     1.0   
resistance-nodulation-cell division (RND) antib...                    14.0   
small multidrug resistance (SMR) antibiotic eff...                     3.0   
sulfonamide resistant sul                                              2.0   

                                                                      \
Organism                                           Enterobacter spp.   
AMR Gene Family                                                        
16S rRNA methyltransferase (G1405)                               3.0   
AAC(3)                                                           5.0   
AAC(6')                                                         13.0   
ANT(2'')                                                         1.0   
ANT(3'')                                                         8.0   
APH(3'')                                                         1.0   
APH(3')                                                          4.0   
APH(6)                                                           1.0   
CARB beta-lactamase                                              1.0   
CMY beta-lactamase                                               2.0   
CTX-M beta-lactamase                                             8.0   
Erm 23S ribosomal RNA methyltransferase                          1.0   
GES beta-lactamase                                               1.0   
IMP beta-lactamase                                               8.0   
MCR phosphoethanolamine transferase                              5.0   
NDM beta-lactamase                                               4.0   
OXA beta-lactamase                                               8.0   
OXA-2-like beta-lactamase                             

In [ ]:
'''
# Save the shared-family list with per-organism detail.
shared_families_detail_gram_negative_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_Shared_AMR_Gene_Families_gm_neg.csv')
)
'''

In [32]:
 gene_family_counts_long_df

,Organism,AMR Gene Family,Count,Percent of Total Families,Percent of Total Genes
0,Acinetobacter baumannii,APH(4),1,2.0,0.269542
1,Acinetobacter baumannii,resistance-nodulation-cell division (RND) anti...,14,28.0,3.773585
2,Acinetobacter baumannii,fluoroquinolone resistant parC,1,2.0,0.269542
3,Acinetobacter baumannii,small multidrug resistance (SMR) antibiotic ef...,3,6.0,0.808625
4,Acinetobacter baumannii,GIM beta-lactamase,1,2.0,0.269542
...,...,...,...,...,...
340,Staphylococcus aureus,daptomycin resistant walK,1,2.0,0.934579
341,Staphylococcus aureus,antibiotic-resistant isoleucyl-tRNA synthetase...,2,4.0,1.869159
342,Staphylococcus aureus,lincosamide nucleotidyltransferase (LNU),3,6.0,2.803738
343,Staphylococcus aureus,vanS,1,2.0,0.934579


In [ ]:
'''
# Save the shared-family list with per-organism detail.
gene_family_counts_long_df.to_csv(
    os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/ESKAPE_all_AMR_Gene_Families.csv')
)
'''

In [34]:
import pandas as pd
df1 = pd.read_csv("/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/AMR_GeneDistribution/1.ESKAPE_all_AMR_Gene_Families.csv")
df1.head(5).style

,Unnamed: 0,Organism,AMR Gene Family,Count,Percent of Total Genes
0,0,Acinetobacter baumannii,APH(4),1,0.269542
1,1,Acinetobacter baumannii,resistance-nodulation-cell division (RND) antibiotic efflux pump,14,3.773585
2,2,Acinetobacter baumannii,fluoroquinolone resistant parC,1,0.269542
3,3,Acinetobacter baumannii,small multidrug resistance (SMR) antibiotic efflux pump,3,0.808625
4,4,Acinetobacter baumannii,GIM beta-lactamase,1,0.269542


In [35]:
df1 = df1.drop('Unnamed: 0', axis=1)

# Display the first few rows to verify the column is gone
df1.head(5).style

,Organism,AMR Gene Family,Count,Percent of Total Genes
0,Acinetobacter baumannii,APH(4),1,0.269542
1,Acinetobacter baumannii,resistance-nodulation-cell division (RND) antibiotic efflux pump,14,3.773585
2,Acinetobacter baumannii,fluoroquinolone resistant parC,1,0.269542
3,Acinetobacter baumannii,small multidrug resistance (SMR) antibiotic efflux pump,3,0.808625
4,Acinetobacter baumannii,GIM beta-lactamase,1,0.269542


In [36]:
grouped_genes = df1.groupby('Organism')['AMR Gene Family'].apply(set)

In [37]:
common_gene_families = set.intersection(*grouped_genes)

In [38]:
# 3. Display the results
print(f"Found {len(common_gene_families)} gene families common to all organisms:")
print(common_gene_families)

Found 7 gene families common to all organisms:
{"APH(3')", 'fosfomycin thiol transferase', 'msr-type ABC-F protein', 'Erm 23S ribosomal RNA methyltransferase', 'chloramphenicol acetyltransferase (CAT)', 'major facilitator superfamily (MFS) antibiotic efflux pump', 'small multidrug resistance (SMR) antibiotic efflux pump'}


## 13. Short names for the 30 gram-negative shared gene families

Maps each of the 30 full family names to a short, FASTA-header-safe code (alphanumeric only -- no spaces, parentheses, or apostrophes), so sequences can be traced back to their family later. Where the family name already contains a standard abbreviation (MFS, RND, SMR, CAT, MPH, qnr), that's reused directly. `FosA` and `Ere` are the field-standard short names for "fosfomycin thiol transferase" and "macrolide esterase". `APH(3'')` and `APH(3')` are genuinely different enzymes (double-prime vs. single-prime), so they get distinct codes (`APH3B`/`APH3A`) rather than colliding on `APH3`.

In [39]:
gram_negative_family_short_names = {
    '16S rRNA methyltransferase (G1405)': '16SMT',
    "AAC(3)": 'AAC3',
    "AAC(6')": 'AAC6',
    "ANT(2'')": 'ANT2',
    "ANT(3'')": 'ANT3',
    "APH(3'')": 'APH3B',
    "APH(3')": 'APH3A',
    'APH(6)': 'APH6',
    'CARB beta-lactamase': 'CARB',
    'CMY beta-lactamase': 'CMY',
    'CTX-M beta-lactamase': 'CTXM',
    'Erm 23S ribosomal RNA methyltransferase': 'ERM',
    'GES beta-lactamase': 'GES',
    'IMP beta-lactamase': 'IMP',
    'MCR phosphoethanolamine transferase': 'MCR',
    'NDM beta-lactamase': 'NDM',
    'OXA beta-lactamase': 'OXA',
    'OXA-2-like beta-lactamase': 'OXA2',
    'TEM beta-lactamase': 'TEM',
    'chloramphenicol acetyltransferase (CAT)': 'CAT',
    'fluoroquinolone resistant gyrA': 'gyrA',
    'fosfomycin thiol transferase': 'FosA',
    'macrolide esterase': 'Ere',
    'macrolide phosphotransferase (MPH)': 'MPH',
    'major facilitator superfamily (MFS) antibiotic efflux pump': 'MFS',
    'msr-type ABC-F protein': 'MSR',
    'quinolone resistance protein (qnr)': 'QNR',
    'resistance-nodulation-cell division (RND) antibiotic efflux pump': 'RND',
    'small multidrug resistance (SMR) antibiotic efflux pump': 'SMR',
    'sulfonamide resistant sul': 'SUL',
}

# Sanity checks: 30 entries, all short codes distinct, keys match the shared-family list exactly.
assert len(gram_negative_family_short_names) == 30
assert len(set(gram_negative_family_short_names.values())) == 30, 'duplicate short codes found'
assert set(gram_negative_family_short_names.keys()) == set(shared_families_gram_negative), \
    'dictionary keys do not exactly match shared_families_gram_negative'

print(f'{len(gram_negative_family_short_names)} gene families mapped to unique short codes.')
gram_negative_family_short_names

30 gene families mapped to unique short codes.


{'16S rRNA methyltransferase (G1405)': '16SMT',
 'AAC(3)': 'AAC3',
 "AAC(6')": 'AAC6',
 "ANT(2'')": 'ANT2',
 "ANT(3'')": 'ANT3',
 "APH(3'')": 'APH3B',
 "APH(3')": 'APH3A',
 'APH(6)': 'APH6',
 'CARB beta-lactamase': 'CARB',
 'CMY beta-lactamase': 'CMY',
 'CTX-M beta-lactamase': 'CTXM',
 'Erm 23S ribosomal RNA methyltransferase': 'ERM',
 'GES beta-lactamase': 'GES',
 'IMP beta-lactamase': 'IMP',
 'MCR phosphoethanolamine transferase': 'MCR',
 'NDM beta-lactamase': 'NDM',
 'OXA beta-lactamase': 'OXA',
 'OXA-2-like beta-lactamase': 'OXA2',
 'TEM beta-lactamase': 'TEM',
 'chloramphenicol acetyltransferase (CAT)': 'CAT',
 'fluoroquinolone resistant gyrA': 'gyrA',
 'fosfomycin thiol transferase': 'FosA',
 'macrolide esterase': 'Ere',
 'macrolide phosphotransferase (MPH)': 'MPH',
 'major facilitator superfamily (MFS) antibiotic efflux pump': 'MFS',
 'msr-type ABC-F protein': 'MSR',
 'quinolone resistance protein (qnr)': 'QNR',
 'resistance-nodulation-cell division (RND) antibiotic efflux pump'

In [86]:
all_beta_lactam_family_short_names = {
    'ACC beta-lactamase': 'ACC',
    'ACT beta-lactamase': 'ACT',
    'ADC beta-lactamase without carbapenemase activity': 'ADCn',
    'ADC beta-lactamases pending classification for carbapenemase activity': 'ADCp',
    'AFM beta-lactamase': 'AFM',
    'BKC Beta-lactamase': 'BKC',
    'CARB beta-lactamase': 'CARB',
    'CMH beta-lactamase': 'CMH',
    'CMY beta-lactamase': 'CMY',
    'CTX-M beta-lactamase': 'CTXM',
    'DHA beta-lactamase': 'DHA',
    'DIM beta-lactamase': 'DIM',
    'FIM beta-lactamase': 'FIM',
    'FONA beta-lactamase': 'FONA',
    'FRI beta-lactamase': 'FRI',
    'GES beta-lactamase': 'GES',
    'GIM beta-lactamase': 'GIM',
    'General Bacterial Porin with reduced permeability to beta-lactams': 'PORIN',
    'IMI beta-lactamase': 'IMI',
    'IMP beta-lactamase': 'IMP',
    'KHM beta-lactamase': 'KHM',
    'KPC beta-lactamase': 'KPC',
    'LAP beta-lactamase': 'LAP',
    'LCR beta-lactamase': 'LCR',
    'LEN beta-lactamase': 'LEN',
    'MIR beta-lactamase': 'MIR',
    'MOX beta-lactamase': 'MOX',
    'NDM beta-lactamase': 'NDM',
    'OXA beta-lactamase': 'OXA',
    'OXA-1-like beta-lactamase': 'OXA1',
    'OXA-10-like beta-lactamase': 'OXA10',
    'OXA-134-like beta-lactamase': 'OXA134',
    'OXA-2-like beta-lactamase': 'OXA2',
    'OXA-23-like beta-lactamase': 'OXA23',
    'OXA-24-like beta-lactamase': 'OXA24',
    'OXA-46-like beta-lactamase': 'OXA46',
    'OXA-48-like beta-lactamase': 'OXA48',
    'OXA-5-like beta-lactamase': 'OXA5',
    'OXA-50-like beta-lactamase': 'OXA50',
    'OXA-51-like beta-lactamase': 'OXA51',
    'OXA-58-like beta-lactamase': 'OXA58',
    'OXA-9-like beta-lactamase': 'OXA9',
    'PDC beta-lactamase': 'PDC',
    'PER beta-lactamase': 'PER',
    'PME beta-lactamase': 'PME',
    'Penicillin-binding protein mutations conferring resistance to beta-lactam antibiotics': 'PBP',
    'RSA beta-lactamase': 'RSA',
    'SCO beta-lactamase': 'SCO',
    'SED beta-lactamase': 'SED',
    'SFO beta-lactamase': 'SFO',
    'SHV beta-lactamase': 'SHV',
    'SIM beta-lactamase': 'SIM',
    'SPM beta-lactamase': 'SPM',
    'TEM beta-lactamase': 'TEM',
    'VEB beta-lactamase': 'VEB',
    'VIM beta-lactamase': 'VIM',
    'ampC-type beta-lactamase': 'ampC',
}

assert len(all_beta_lactam_family_short_names) == 57
assert len(set(all_beta_lactam_family_short_names.values())) == 57, 'duplicate short codes found'

# Organism code -> full (non-gram-negative-restricted) ARG_Fasta_Files folder name.
BETA_LACTAM_ORG_FOLDERS = {
    'AB':  'AB_ARG_Fasta_Files',
    'ENT': 'ENT_ARG_Fasta_Files',
    'Kp':  'Kp_ARG_Fasta_Files',
    'Pa':  'Pa_ARG_Fasta_Files',
}
BETA_LACTAM_OUTPUT_ROOT = os.path.join(GENEFAMILY_DIR, 'All_Beta_Lactam_Relabeled')

for org, folder in BETA_LACTAM_ORG_FOLDERS.items():
    input_dir = os.path.join(GENEFAMILY_DIR, folder)
    output_dir = os.path.join(BETA_LACTAM_OUTPUT_ROOT, org)
    os.makedirs(output_dir, exist_ok=True)

    org_counts = {}
    for family_name, short_code in all_beta_lactam_family_short_names.items():
        input_path = os.path.join(input_dir, f'{family_name}.fasta')
        if not os.path.exists(input_path):
            continue  # this organism doesn't have this family -- skip, don't crash
        output_path = os.path.join(output_dir, f'{family_name}.fasta')
        org_counts[family_name] = relabel_fasta_headers(input_path, output_path, org, short_code)

    print(f'{org}: {len(org_counts)} beta-lactam families relabeled, {sum(org_counts.values())} total records')


AB: 20 beta-lactam families relabeled, 5772 total records
ENT: 34 beta-lactam families relabeled, 4274 total records
Kp: 29 beta-lactam families relabeled, 28571 total records
Pa: 27 beta-lactam families relabeled, 5522 total records


## 14. Relabel FASTA headers for traceability (30 gram-negative shared families)

Rewrites each sequence's header in the 30-family gram-negative subset (`Results/Genefamily/GN_AMR_Family/<Org>/`, `AB`/`ENT`/`Kp`/`Pa`) from the bare gene name (e.g. `>armA`) to `gene_org_shortcode_index` (e.g. `>armA_AB_16SMT_1`), using the `gram_negative_family_short_names` dictionary from section 13.

**Why the trailing index:** gene names repeat heavily within one family/organism file, and the repeats are not always the same sequence -- e.g. AB's `armA` appears 424 times across 6 distinct sequence variants, and Kp's `CTX-M-15` appears 1837 times across 11 variants. Appending only `_org_shortcode` would collapse all of those onto one duplicate header, which breaks downstream alignment tools (MAFFT/PAML both require unique sequence IDs) and erases the very traceability this is meant to add. The index is a 1-based counter over the records in that file, so headers are unique within the file (and, since organism + family are embedded, unique across the whole 120-file corpus).

Output goes to a new directory (`GN_AMR_Family_Relabeled`) -- the originals in `GN_AMR_Family/` and `<Org>_ARG_Fasta_Files/` are read-only inputs here, never modified.

### iF REQUIRED TO RERUN CHANGE THE DIRECTORY TO GN

In [82]:
# Section 14 config: relabel the 30-family gram-negative subset into a new output directory.
GN_INPUT_ROOT = os.path.join(GENEFAMILY_DIR, 'ENT_ARG_Fasta_Files')
GN_OUTPUT_ROOT = os.path.join(GENEFAMILY_DIR, 'ENT_ARG_Fasta_Files_Relabeled')
GN_RELABEL_ORGS = ['AB', 'ENT', 'Kp', 'Pa']

### 14.1 Shared helpers

Pure file I/O (parsing/rewriting FASTA records) -- not the analysis itself.

In [83]:
def relabel_fasta_headers(input_path, output_path, org_code, family_short_code):
    """
    Rewrite one gene-family FASTA's headers as '>{gene}_{org}_{code}_{i}' (1-based, per-file
    counter), to disambiguate the many repeated gene names within a family while keeping every
    original sequence byte-for-byte unchanged. Returns the number of records written.
    """
    records = []
    gene_name = None
    seq_lines = []
    with open(input_path) as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('>'):
                if gene_name is not None:
                    records.append((gene_name, seq_lines))
                gene_name = line[1:].strip()
                seq_lines = []
            else:
                seq_lines.append(line)
        if gene_name is not None:
            records.append((gene_name, seq_lines))

    with open(output_path, 'w') as out:
        for i, (gene, seq) in enumerate(records, start=1):
            out.write(f'>{gene}_{org_code}_{family_short_code}_{i}\n')
            for seq_line in seq:
                out.write(seq_line + '\n')

    return len(records)

In [84]:
def read_fasta_records(path):
    """Parse a FASTA file into a list of (header, full_sequence) tuples, for verification."""
    records = []
    header = None
    seq_lines = []
    with open(path) as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('>'):
                if header is not None:
                    records.append((header, ''.join(seq_lines)))
                header = line[1:].strip()
                seq_lines = []
            else:
                seq_lines.append(line)
        if header is not None:
            records.append((header, ''.join(seq_lines)))
    return records

### 14.2 Per-organism relabeling

One explicit cell per organism, each producing its own named variable -- no loop over organisms, consistent with this notebook's per-organism cells elsewhere.

In [43]:
# AB: relabel all 30 gram-negative shared-family FASTA files.
AB_relabel_input_dir = os.path.join(GN_INPUT_ROOT, 'AB')
AB_relabel_output_dir = os.path.join(GN_OUTPUT_ROOT, 'AB')
os.makedirs(AB_relabel_output_dir, exist_ok=True)

AB_relabel_counts = {}
for family_name, short_code in gram_negative_family_short_names.items():
    input_path = os.path.join(AB_relabel_input_dir, f'{family_name}.fasta')
    output_path = os.path.join(AB_relabel_output_dir, f'{family_name}.fasta')
    AB_relabel_counts[family_name] = relabel_fasta_headers(input_path, output_path, 'AB', short_code)

print(f'AB: {len(AB_relabel_counts)} files relabeled, {sum(AB_relabel_counts.values())} total records')

AB: 30 files relabeled, 27359 total records


In [85]:
# ENT: relabel all 30 gram-negative shared-family FASTA files.
#ENT_relabel_input_dir = os.path.join(GN_INPUT_ROOT, 'ENT')
ENT_relabel_input_dir = os.path.join(GN_INPUT_ROOT)
#ENT_relabel_output_dir = os.path.join(GN_OUTPUT_ROOT, 'ENT')
ENT_relabel_output_dir = os.path.join(GN_OUTPUT_ROOT)
os.makedirs(ENT_relabel_output_dir, exist_ok=True)

ENT_relabel_counts = {}
for family_name, short_code in gram_negative_family_short_names.items():
    input_path = os.path.join(ENT_relabel_input_dir, f'{family_name}.fasta')
    output_path = os.path.join(ENT_relabel_output_dir, f'{family_name}.fasta')
    ENT_relabel_counts[family_name] = relabel_fasta_headers(input_path, output_path, 'ENT', short_code)

print(f'ENT: {len(ENT_relabel_counts)} files relabeled, {sum(ENT_relabel_counts.values())} total records')

ENT: 30 files relabeled, 19620 total records


In [45]:
# Kp: relabel all 30 gram-negative shared-family FASTA files.
Kp_relabel_input_dir = os.path.join(GN_INPUT_ROOT, 'Kp')
Kp_relabel_output_dir = os.path.join(GN_OUTPUT_ROOT, 'Kp')
os.makedirs(Kp_relabel_output_dir, exist_ok=True)

Kp_relabel_counts = {}
for family_name, short_code in gram_negative_family_short_names.items():
    input_path = os.path.join(Kp_relabel_input_dir, f'{family_name}.fasta')
    output_path = os.path.join(Kp_relabel_output_dir, f'{family_name}.fasta')
    Kp_relabel_counts[family_name] = relabel_fasta_headers(input_path, output_path, 'Kp', short_code)

print(f'Kp: {len(Kp_relabel_counts)} files relabeled, {sum(Kp_relabel_counts.values())} total records')

Kp: 30 files relabeled, 114394 total records


In [46]:
# Pa: relabel all 30 gram-negative shared-family FASTA files.
Pa_relabel_input_dir = os.path.join(GN_INPUT_ROOT, 'Pa')
Pa_relabel_output_dir = os.path.join(GN_OUTPUT_ROOT, 'Pa')
os.makedirs(Pa_relabel_output_dir, exist_ok=True)

Pa_relabel_counts = {}
for family_name, short_code in gram_negative_family_short_names.items():
    input_path = os.path.join(Pa_relabel_input_dir, f'{family_name}.fasta')
    output_path = os.path.join(Pa_relabel_output_dir, f'{family_name}.fasta')
    Pa_relabel_counts[family_name] = relabel_fasta_headers(input_path, output_path, 'Pa', short_code)

print(f'Pa: {len(Pa_relabel_counts)} files relabeled, {sum(Pa_relabel_counts.values())} total records')

Pa: 30 files relabeled, 63901 total records


### 14.3 Verification

For every (organism, family) file: record count matches the input exactly, every header within the file is unique, and sequence content is byte-identical to the input -- only headers changed.

In [47]:
GN_RELABEL_COUNTS = {'AB': AB_relabel_counts, 'ENT': ENT_relabel_counts, 'Kp': Kp_relabel_counts, 'Pa': Pa_relabel_counts}

all_checks_passed = True
for org in GN_RELABEL_ORGS:
    for family_name in gram_negative_family_short_names:
        input_path = os.path.join(GN_INPUT_ROOT, org, f'{family_name}.fasta')
        output_path = os.path.join(GN_OUTPUT_ROOT, org, f'{family_name}.fasta')
        input_records = read_fasta_records(input_path)
        output_records = read_fasta_records(output_path)

        if len(input_records) != len(output_records):
            print(f'FAIL count mismatch: {org} / {family_name}: input={len(input_records)} output={len(output_records)}')
            all_checks_passed = False
            continue

        output_headers = [rec[0] for rec in output_records]
        if len(set(output_headers)) != len(output_headers):
            print(f'FAIL duplicate headers: {org} / {family_name}')
            all_checks_passed = False

        input_seqs = [rec[1] for rec in input_records]
        output_seqs = [rec[1] for rec in output_records]
        if input_seqs != output_seqs:
            print(f'FAIL sequence mismatch: {org} / {family_name}')
            all_checks_passed = False

print('All checks passed: record counts match, headers unique per file, sequences unchanged.'
      if all_checks_passed else 'Some checks FAILED -- see above.')

All checks passed: record counts match, headers unique per file, sequences unchanged.


In [48]:
# Spot-check: AB armA record #1.
with open(os.path.join(GN_OUTPUT_ROOT, 'AB', "16S rRNA methyltransferase (G1405).fasta")) as f:
    print(f.readline().strip())
    print(f.readline().strip()[:60], '...')

>armA_AB_16SMT_1
ATGGATAAGAATGATGTTGTTAAGAAGATACTTGAATCAAAAAAGTACGAAAACCTTGAT ...


## 15. Gene counts and sequence lengths per family (relabeled gram-negative subset)

For each of the 30 gram-negative shared families, how many distinct genes/alleles are present, and what is their sequence length -- the length-consistency check flagged earlier as a prerequisite before aligning a gene: within one gene name, sequences should be similar length; large spread signals a partial hit, an indel variant, or an assembly artifact worth excluding before alignment.

Reads directly from the relabeled folder (`GN_AMR_Family_Relabeled`, section 14's output) and recovers the original gene name from each header (`gene_org_shortcode_index`).

In [49]:
def parse_relabeled_fasta_gene_lengths(path):
    """
    Parse a relabeled FASTA file, recovering the original gene name from each
    header ('gene_org_shortcode_index') and the sequence length for that record.
    Returns a list of (gene_name, length) tuples, one per record.
    """
    records = []
    gene_name = None
    seq_len = 0
    with open(path) as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('>'):
                if gene_name is not None:
                    records.append((gene_name, seq_len))
                parts = line[1:].strip().split('_')
                gene_name = '_'.join(parts[:-3])  # strip trailing org, shortcode, index
                seq_len = 0
            else:
                seq_len += len(line)
        if gene_name is not None:
            records.append((gene_name, seq_len))
    return records

### 15.1 Per-organism gene + length extraction

One explicit cell per organism, consistent with this notebook's per-organism cells elsewhere.

In [50]:
# AB: gene name + sequence length per record, across all 30 gram-negative shared families.
AB_gene_length_rows = []
for family_name in gram_negative_family_short_names:
    path = os.path.join(GN_OUTPUT_ROOT, 'AB', f'{family_name}.fasta')
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        AB_gene_length_rows.append(
            {'Organism': 'AB', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

AB_gene_length_df = pd.DataFrame(AB_gene_length_rows)
print(f'AB: {len(AB_gene_length_df)} records, {AB_gene_length_df["Gene"].nunique()} distinct genes')

AB: 27359 records, 172 distinct genes


In [51]:
# ENT: gene name + sequence length per record, across all 30 gram-negative shared families.
ENT_gene_length_rows = []
for family_name in gram_negative_family_short_names:
    path = os.path.join(GN_OUTPUT_ROOT, 'ENT', f'{family_name}.fasta')
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        ENT_gene_length_rows.append(
            {'Organism': 'ENT', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

ENT_gene_length_df = pd.DataFrame(ENT_gene_length_rows)
print(f'ENT: {len(ENT_gene_length_df)} records, {ENT_gene_length_df["Gene"].nunique()} distinct genes')

ENT: 19620 records, 161 distinct genes


In [52]:
# Kp: gene name + sequence length per record, across all 30 gram-negative shared families.
Kp_gene_length_rows = []
for family_name in gram_negative_family_short_names:
    path = os.path.join(GN_OUTPUT_ROOT, 'Kp', f'{family_name}.fasta')
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        Kp_gene_length_rows.append(
            {'Organism': 'Kp', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

Kp_gene_length_df = pd.DataFrame(Kp_gene_length_rows)
print(f'Kp: {len(Kp_gene_length_df)} records, {Kp_gene_length_df["Gene"].nunique()} distinct genes')

Kp: 114394 records, 263 distinct genes


In [53]:
# Pa: gene name + sequence length per record, across all 30 gram-negative shared families.
Pa_gene_length_rows = []
for family_name in gram_negative_family_short_names:
    path = os.path.join(GN_OUTPUT_ROOT, 'Pa', f'{family_name}.fasta')
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        Pa_gene_length_rows.append(
            {'Organism': 'Pa', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

Pa_gene_length_df = pd.DataFrame(Pa_gene_length_rows)
print(f'Pa: {len(Pa_gene_length_df)} records, {Pa_gene_length_df["Gene"].nunique()} distinct genes')

Pa: 63901 records, 244 distinct genes


### 15.2 Combine across organisms

In [54]:
gene_length_long_df = pd.concat(
    [AB_gene_length_df, ENT_gene_length_df, Kp_gene_length_df, Pa_gene_length_df],
    ignore_index=True,
)
print(f'{len(gene_length_long_df)} total records across all 4 organisms')
gene_length_long_df.head()

225274 total records across all 4 organisms


,Organism,AMR Gene Family,Gene,Length
0,AB,16S rRNA methyltransferase (G1405),armA,774
1,AB,16S rRNA methyltransferase (G1405),armA,774
2,AB,16S rRNA methyltransferase (G1405),armA,774
3,AB,16S rRNA methyltransferase (G1405),armA,774
4,AB,16S rRNA methyltransferase (G1405),armA,774


In [ ]:
'''
# Save the shared-family list with per-organism detail.
gene_length_long_df.to_csv(
    os.path.join(BASE_DIR, 'Results/Genefamily/GN_AMR_Family_Relabeled/Gene_length_distribution_2.csv')
)
'''

### 15.3 Per-gene summary (pooled across organisms) -- answers "what is the gene length"

In [55]:
gene_length_summary_df = (
    gene_length_long_df
    .groupby(['AMR Gene Family', 'Gene'])
    .agg(
        Organisms_Present=('Organism', lambda s: ', '.join(sorted(s.unique()))),
        N_Organisms=('Organism', 'nunique'),
        N_Records=('Gene', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
        N_Distinct_Lengths=('Length', 'nunique'),
    )
    .reset_index()
    .sort_values(['AMR Gene Family', 'Gene'])
)
gene_length_summary_df

,AMR Gene Family,Gene,Organisms_Present,N_Organisms,N_Records,Min_Length,Max_Length,N_Distinct_Lengths
0,16S rRNA methyltransferase (G1405),armA,"AB, ENT, Kp, Pa",4,688,690,816,3
1,16S rRNA methyltransferase (G1405),rmtB,"ENT, Kp, Pa",3,580,690,858,4
2,16S rRNA methyltransferase (G1405),rmtC,"AB, ENT, Kp",3,13,846,846,1
3,16S rRNA methyltransferase (G1405),rmtD,Pa,1,5,744,744,1
4,16S rRNA methyltransferase (G1405),rmtF,"Kp, Pa",2,142,780,780,1
...,...,...,...,...,...,...,...,...
561,small multidrug resistance (SMR) antibiotic ef...,qacJ,"AB, ENT, Kp",3,69,204,1071,19
562,small multidrug resistance (SMR) antibiotic ef...,qacL,"ENT, Kp, Pa",3,159,333,333,1
563,sulfonamide resistant sul,sul1,"AB, ENT, Kp, Pa",4,4395,780,1185,15
564,sulfonamide resistant sul,sul2,"AB, ENT, Kp, Pa",4,2177,816,897,12


### 15.4 Per-family summary -- answers "how many genes are present in each family"

In [56]:
family_gene_count_df = (
    gene_length_summary_df
    .groupby('AMR Gene Family')
    .agg(N_Distinct_Genes=('Gene', 'nunique'))
    .reset_index()
    .sort_values('N_Distinct_Genes', ascending=False)
)
family_gene_count_df

,AMR Gene Family,N_Distinct_Genes
16,OXA beta-lactamase,173
27,resistance-nodulation-cell division (RND) anti...,71
24,major facilitator superfamily (MFS) antibiotic...,36
10,CTX-M beta-lactamase,33
18,TEM beta-lactamase,27
26,quinolone resistance protein (qnr),24
13,IMP beta-lactamase,23
2,AAC(6'),23
4,ANT(3''),20
14,MCR phosphoethanolamine transferase,19


## 16. Per-organism breakdowns: gene counts and gene lengths

Section 15 pooled genes/lengths across all 4 organisms. This section keeps the organism dimension: how many distinct genes each family has *within each organism*, and the sequence-length summary per (organism, family, gene) -- reused directly from `gene_length_long_df` (section 15.2), just grouped differently.

In [57]:
def genes_per_family_per_organism(long_df):
    """Distinct gene count per (AMR Gene Family, Organism), pivoted wide (organism = column)."""
    return (
        long_df
        .groupby(['AMR Gene Family', 'Organism'])['Gene']
        .nunique()
        .unstack('Organism')
        .fillna(0)
        .astype(int)
        .reindex(columns=GN_RELABEL_ORGS)
    )

genes_per_family_per_organism_df = genes_per_family_per_organism(gene_length_long_df)
genes_per_family_per_organism_df

Organism,AB,ENT,Kp,Pa
AMR Gene Family,,,,
16S rRNA methyltransferase (G1405),2,3,6,4
AAC(3),5,5,7,3
AAC(6'),4,13,18,8
ANT(2''),1,1,1,1
ANT(3''),8,8,12,12
APH(3''),1,1,1,1
APH(3'),3,4,6,6
APH(6),1,1,2,2
CARB beta-lactamase,3,1,1,3


In [58]:
def gene_length_per_family_per_organism(long_df):
    """Sequence-length summary per (Organism, AMR Gene Family, Gene)."""
    return (
        long_df
        .groupby(['Organism', 'AMR Gene Family', 'Gene'])
        .agg(
            N_Records=('Length', 'size'),
            Min_Length=('Length', 'min'),
            Max_Length=('Length', 'max'),
            N_Distinct_Lengths=('Length', 'nunique'),
        )
        .reset_index()
        .sort_values(['AMR Gene Family', 'Gene', 'Organism'])
    )

gene_length_per_organism_df = gene_length_per_family_per_organism(gene_length_long_df)
gene_length_per_organism_df

,Organism,AMR Gene Family,Gene,N_Records,Min_Length,Max_Length,N_Distinct_Lengths
0,AB,16S rRNA methyltransferase (G1405),armA,424,690,816,3
173,ENT,16S rRNA methyltransferase (G1405),armA,24,774,774,1
336,Kp,16S rRNA methyltransferase (G1405),armA,237,774,774,1
601,Pa,16S rRNA methyltransferase (G1405),armA,3,774,774,1
174,ENT,16S rRNA methyltransferase (G1405),rmtB,8,756,756,1
...,...,...,...,...,...,...,...
334,ENT,sulfonamide resistant sul,sul2,132,816,867,4
599,Kp,sulfonamide resistant sul,sul2,1535,816,897,7
849,Pa,sulfonamide resistant sul,sul2,9,816,852,2
335,ENT,sulfonamide resistant sul,sul3,5,792,792,1


In [59]:
'''
# Save the shared-family list with per-organism detail.
gene_length_per_organism_df.to_csv(
    os.path.join(BASE_DIR, 'Results/Genefamily/GN_AMR_Family_Relabeled/Gene_length_distribution.csv')
)
'''

"\n# Save the shared-family list with per-organism detail.\ngene_length_per_organism_df.to_csv(\n    os.path.join(BASE_DIR, 'Results/Genefamily/GN_AMR_Family_Relabeled/Gene_length_distribution.csv')\n)\n"

## Section 17

## 17. Metadata for the CD-HIT-EST deduplicated sequences (100% identity)

Reruns sections 15-16's metadata build on the CD-HIT-EST representative sequences (`Results/Genefamily/Dedup_03_08_26/`) instead of the full relabeled set -- same 30 gram-negative shared families, same 4 organisms, headers untouched by CD-HIT-EST so the same parsing/summary functions apply directly. Only the input paths change.

In [114]:
DEDUP_ROOT = os.path.join(GENEFAMILY_DIR, 'Dedup_03_08_26')
DEDUP_ORG_FOLDERS = {
    'AB':  'AB_ Representative sequences',
    'ENT': 'ENT',
    'Kp':  'KP',
    'Pa':  'PA',
}

### 17.1 Per-organism gene + length extraction (deduplicated)

Same pattern as section 15.1, reusing `parse_relabeled_fasta_gene_lengths` (section 14.1) unchanged -- only the file path (dedup root/folder, doubled `.fasta.fasta` extension) differs.

In [115]:
# AB (deduplicated): gene name + sequence length per representative record.
AB_dedup_gene_length_rows = []
for family_name in gram_negative_family_short_names:
    path = os.path.join(DEDUP_ROOT, DEDUP_ORG_FOLDERS['AB'], f'{family_name}.fasta.fasta')
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        AB_dedup_gene_length_rows.append(
            {'Organism': 'AB', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

AB_dedup_gene_length_df = pd.DataFrame(AB_dedup_gene_length_rows)
print(f'AB (deduplicated): {len(AB_dedup_gene_length_df)} records, {AB_dedup_gene_length_df["Gene"].nunique()} distinct genes')

AB (deduplicated): 1877 records, 171 distinct genes


In [62]:
# ENT (deduplicated): gene name + sequence length per representative record.
ENT_dedup_gene_length_rows = []
for family_name in gram_negative_family_short_names:
    path = os.path.join(DEDUP_ROOT, DEDUP_ORG_FOLDERS['ENT'], f'{family_name}.fasta.fasta')
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        ENT_dedup_gene_length_rows.append(
            {'Organism': 'ENT', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

ENT_dedup_gene_length_df = pd.DataFrame(ENT_dedup_gene_length_rows)
print(f'ENT (deduplicated): {len(ENT_dedup_gene_length_df)} records, {ENT_dedup_gene_length_df["Gene"].nunique()} distinct genes')

ENT (deduplicated): 4152 records, 157 distinct genes


In [63]:
# Kp (deduplicated): gene name + sequence length per representative record.
Kp_dedup_gene_length_rows = []
for family_name in gram_negative_family_short_names:
    path = os.path.join(DEDUP_ROOT, DEDUP_ORG_FOLDERS['Kp'], f'{family_name}.fasta.fasta')
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        Kp_dedup_gene_length_rows.append(
            {'Organism': 'Kp', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

Kp_dedup_gene_length_df = pd.DataFrame(Kp_dedup_gene_length_rows)
print(f'Kp (deduplicated): {len(Kp_dedup_gene_length_df)} records, {Kp_dedup_gene_length_df["Gene"].nunique()} distinct genes')

Kp (deduplicated): 5397 records, 256 distinct genes


In [64]:
# Pa (deduplicated): gene name + sequence length per representative record.
Pa_dedup_gene_length_rows = []
for family_name in gram_negative_family_short_names:
    path = os.path.join(DEDUP_ROOT, DEDUP_ORG_FOLDERS['Pa'], f'{family_name}.fasta.fasta')
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        Pa_dedup_gene_length_rows.append(
            {'Organism': 'Pa', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

Pa_dedup_gene_length_df = pd.DataFrame(Pa_dedup_gene_length_rows)
print(f'Pa (deduplicated): {len(Pa_dedup_gene_length_df)} records, {Pa_dedup_gene_length_df["Gene"].nunique()} distinct genes')

Pa (deduplicated): 8788 records, 241 distinct genes


### 17.2 Combine across organisms (deduplicated equivalent of `Gene_length_distribution_2.csv`)

In [65]:
gene_length_long_dedup_df = pd.concat(
    [AB_dedup_gene_length_df, ENT_dedup_gene_length_df, Kp_dedup_gene_length_df, Pa_dedup_gene_length_df],
    ignore_index=True,
)
print(f'{len(gene_length_long_dedup_df)} total deduplicated records across all 4 organisms')
gene_length_long_dedup_df.head()

20214 total deduplicated records across all 4 organisms


,Organism,AMR Gene Family,Gene,Length
0,AB,16S rRNA methyltransferase (G1405),armA,816
1,AB,16S rRNA methyltransferase (G1405),armA,774
2,AB,16S rRNA methyltransferase (G1405),armA,690
3,AB,16S rRNA methyltransferase (G1405),armA,774
4,AB,16S rRNA methyltransferase (G1405),armA,774


In [70]:
# Save the shared-family list with per-organism detail.
gene_length_long_dedup_df.to_csv(
    os.path.join(BASE_DIR, 'Results/Genefamily/Dedup_03_08_26/Gene_length_distribution.csv')
)

### 17.3 Per-(Organism, Family, Gene) summary (deduplicated equivalent of `Gene_length_distribution.csv`)

Reuses `gene_length_per_family_per_organism` (section 16) unchanged.

In [66]:
gene_length_per_organism_dedup_df = gene_length_per_family_per_organism(gene_length_long_dedup_df)
gene_length_per_organism_dedup_df

,Organism,AMR Gene Family,Gene,N_Records,Min_Length,Max_Length,N_Distinct_Lengths
0,AB,16S rRNA methyltransferase (G1405),armA,5,690,816,3
172,ENT,16S rRNA methyltransferase (G1405),armA,2,774,774,1
331,Kp,16S rRNA methyltransferase (G1405),armA,8,774,774,1
589,Pa,16S rRNA methyltransferase (G1405),armA,1,774,774,1
173,ENT,16S rRNA methyltransferase (G1405),rmtB,3,756,756,1
...,...,...,...,...,...,...,...
329,ENT,sulfonamide resistant sul,sul2,8,816,867,3
587,Kp,sulfonamide resistant sul,sul2,19,816,897,5
834,Pa,sulfonamide resistant sul,sul2,2,816,852,2
330,ENT,sulfonamide resistant sul,sul3,1,792,792,1


In [71]:
# Save the shared-family list with per-organism detail.
gene_length_per_organism_dedup_df.to_csv(
    os.path.join(BASE_DIR, 'Results/Genefamily/Dedup_03_08_26/Gene_length_distribution_2.csv')
)

### 17.4 Pooled per-gene summary across organisms (deduplicated equivalent of section 15.3)

In [67]:
gene_length_summary_dedup_df = (
    gene_length_long_dedup_df
    .groupby(['AMR Gene Family', 'Gene'])
    .agg(
        Organisms_Present=('Organism', lambda s: ', '.join(sorted(s.unique()))),
        N_Organisms=('Organism', 'nunique'),
        N_Records=('Gene', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
        N_Distinct_Lengths=('Length', 'nunique'),
    )
    .reset_index()
    .sort_values(['AMR Gene Family', 'Gene'])
)
gene_length_summary_dedup_df

,AMR Gene Family,Gene,Organisms_Present,N_Organisms,N_Records,Min_Length,Max_Length,N_Distinct_Lengths
0,16S rRNA methyltransferase (G1405),armA,"AB, ENT, Kp, Pa",4,16,690,816,3
1,16S rRNA methyltransferase (G1405),rmtB,"ENT, Kp, Pa",3,13,690,858,4
2,16S rRNA methyltransferase (G1405),rmtC,"AB, ENT, Kp",3,3,846,846,1
3,16S rRNA methyltransferase (G1405),rmtD,Pa,1,2,744,744,1
4,16S rRNA methyltransferase (G1405),rmtF,"Kp, Pa",2,4,780,780,1
...,...,...,...,...,...,...,...,...
554,small multidrug resistance (SMR) antibiotic ef...,qacJ,"AB, ENT, Kp",3,25,234,1071,15
555,small multidrug resistance (SMR) antibiotic ef...,qacL,"ENT, Kp, Pa",3,10,333,333,1
556,sulfonamide resistant sul,sul1,"AB, ENT, Kp, Pa",4,68,780,1185,14
557,sulfonamide resistant sul,sul2,"AB, ENT, Kp, Pa",4,44,816,897,8


### 17.5 Per-family gene count, deduplicated (equivalent of section 15.4)

In [68]:
family_gene_count_dedup_df = (
    gene_length_summary_dedup_df
    .groupby('AMR Gene Family')
    .agg(N_Distinct_Genes=('Gene', 'nunique'))
    .reset_index()
    .sort_values('N_Distinct_Genes', ascending=False)
)
family_gene_count_dedup_df

,AMR Gene Family,N_Distinct_Genes
16,OXA beta-lactamase,173
27,resistance-nodulation-cell division (RND) anti...,71
24,major facilitator superfamily (MFS) antibiotic...,36
10,CTX-M beta-lactamase,32
18,TEM beta-lactamase,27
13,IMP beta-lactamase,23
26,quinolone resistance protein (qnr),22
2,AAC(6'),21
14,MCR phosphoethanolamine transferase,19
4,ANT(3''),19


### 17.6 Per-organism gene-count matrix, deduplicated (equivalent of section 16 first cell)

Reuses `genes_per_family_per_organism` (section 16) unchanged.

In [69]:
genes_per_family_per_organism_dedup_df = genes_per_family_per_organism(gene_length_long_dedup_df)
genes_per_family_per_organism_dedup_df

Organism,AB,ENT,Kp,Pa
AMR Gene Family,,,,
16S rRNA methyltransferase (G1405),2,3,6,4
AAC(3),5,5,7,3
AAC(6'),4,11,16,7
ANT(2''),1,1,1,1
ANT(3''),8,7,11,10
APH(3''),1,1,1,1
APH(3'),3,4,6,6
APH(6),1,1,2,2
CARB beta-lactamase,3,1,1,3


### 17.7 Per-family length summary, deduplicated -- no gene breakdown

Same `gene_length_long_dedup_df` as 17.2, grouped by family only (gene and organism dropped) -- just record count and length range per family.

In [73]:
family_length_summary_dedup_df = (
    gene_length_long_dedup_df
    .groupby('AMR Gene Family')
    .agg(
        N_Records=('Length', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
    )
    .reset_index()
    .sort_values('AMR Gene Family')
)
family_length_summary_dedup_df

,AMR Gene Family,N_Records,Min_Length,Max_Length
0,16S rRNA methyltransferase (G1405),40,690,858
1,AAC(3),55,465,954
2,AAC(6'),129,405,1374
3,ANT(2''),21,486,1035
4,ANT(3''),185,507,1674
5,APH(3''),43,792,1509
6,APH(3'),261,738,861
7,APH(6),47,759,960
8,CARB beta-lactamase,10,867,1041
9,CMY beta-lactamase,19,873,1146


In [ ]:
Organism 	AMR Gene Family 	Count 	Unnamed: 3 	Unnamed: 4
0 	Acinetobacter baumannii 	APH(4) 	1 	NaN 	NaN
1 	Acinetobacter baumannii 	resistance-nodulation-cell division (RND) anti... 	14 	NaN 	NaN
2 	Acinetobacter baumannii 	fluoroquinolone resistant parC 	1 	NaN 	NaN
3 	Acinetobacter baumannii 	small multidrug resistance (SMR) antibiotic ef... 	3 	NaN 	NaN
4 	Acinetobacter baumannii 	GIM beta-lactamase 	1 	NaN 	NaN
... 	... 	... 	... 	... 	...
340 	Staphylococcus aureus 	daptomycin resistant walK 	1 	NaN 	NaN
341 	Staphylococcus aureus 	antibiotic-resistant isoleucyl-tRNA synthetase... 	2 	NaN 	NaN
342 	Staphylococcus aureus 	lincosamide nucleotidyltransferase (LNU) 	3 	NaN 	NaN
343 	Staphylococcus aureus 	vanS 	1 	NaN 	NaN
344 	Staphylococcus aureus 	streptothricin acetyltransferase (SAT) 	1 	NaN 	NaN

### 17.8 Per-organism, per-family length summary, deduplicated -- corrects 17.7

17.7 pooled across organisms entirely. This keeps the organism split (only the gene symbol is dropped): N_Records, Min_Length, Max_Length per (Organism, AMR Gene Family).

In [74]:
organism_family_length_summary_dedup_df = (
    gene_length_long_dedup_df
    .groupby(['Organism', 'AMR Gene Family'])
    .agg(
        N_Records=('Length', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
    )
    .reset_index()
    .sort_values(['AMR Gene Family', 'Organism'])
)
organism_family_length_summary_dedup_df

,Organism,AMR Gene Family,N_Records,Min_Length,Max_Length
0,AB,16S rRNA methyltransferase (G1405),6,690,846
30,ENT,16S rRNA methyltransferase (G1405),6,756,846
60,Kp,16S rRNA methyltransferase (G1405),21,690,858
90,Pa,16S rRNA methyltransferase (G1405),7,744,780
1,AB,AAC(3),11,534,954
...,...,...,...,...,...
118,Pa,small multidrug resistance (SMR) antibiotic ef...,57,333,333
29,AB,sulfonamide resistant sul,26,816,1182
59,ENT,sulfonamide resistant sul,23,792,1182
89,Kp,sulfonamide resistant sul,48,780,1185


In [75]:
# Save the shared-family list with per-organism detail.
organism_family_length_summary_dedup_df.to_csv(
    os.path.join(BASE_DIR, 'Results/Genefamily/Dedup_03_08_26/Gene_length_distribution_3.csv')
)

## 18. CDS (nucleotide) alignment: OXA beta-lactamase only

Focused follow-up to the beta-lactamase pilot: `OXA beta-lactamase` alone (not `OXA-2-like beta-lactamase`, a separate CARD family) -- the family with by far the most gene diversity (173 distinct genes) and the only one that fragmented across many clades in that tree.

This time the alignment is at the **CDS/nucleotide level, not protein** -- the pilot's protein alignment was right for the exploratory clustering question, but codon-based selection-pressure analysis (dN/dS, PAML `codeml`) needs a nucleotide alignment that preserves codon structure, so no translation step here.

In [104]:
# Section 18 config: OXA-only, CDS-level pilot -- reuses DEDUP_ROOT/DEDUP_ORG_FOLDERS (section 17)
# and GN_RELABEL_ORGS (section 14) unchanged.
Beta_lactam_DIR = os.path.join(BASE_DIR, 'Results/Phylogenetics/Beta_lactamase_all')
os.makedirs(Beta_lactam_DIR, exist_ok=True)

OXA_FAMILY = 'OXA beta-lactamase'
MAFFT_BIN = '/home/user/miniforge3/envs/blast/bin/mafft'
FASTTREE_BIN = '/home/user/miniforge3/envs/blast/bin/FastTree'

### 18.1 Pool OXA sequences across the 4 organisms

Reuses `read_fasta_records` (section 14.1) -- no new parsing helper needed.

In [77]:
beta_lactam_pooled_nt_path = os.path.join(Beta_lactam_DIR, 'beta_lactam_pooled_nt.fasta')
beta_lactam_pooled_counts = {}
with open(oxa_pooled_nt_path, 'w') as out:
    for org in GN_RELABEL_ORGS:
        path = os.path.join(DEDUP_ROOT, DEDUP_ORG_FOLDERS[org], f'{OXA_FAMILY}.fasta.fasta')
        records = read_fasta_records(path)
        for header, seq in records:
            out.write(f'>{header}\n{seq}\n')
        oxa_pooled_counts[org] = len(records)

print('Per-organism record counts:', oxa_pooled_counts)
print('Total pooled:', sum(oxa_pooled_counts.values()))

Per-organism record counts: {'AB': 116, 'ENT': 8, 'Kp': 34, 'Pa': 163}
Total pooled: 321


### 18.2 Frame-integrity check

Every sequence must be in-frame (length a multiple of 3) and free of internal stop codons (bacterial genetic code, table 11) before alignment -- failures are excluded rather than passed through.

In [78]:
from Bio.Seq import Seq

oxa_frame_ok = []
oxa_frame_bad = []
for header, seq in read_fasta_records(oxa_pooled_nt_path):
    seq = seq.upper()
    if len(seq) % 3 != 0:
        oxa_frame_bad.append((header, 'bad_length', len(seq)))
        continue
    protein = str(Seq(seq).translate(table=11, to_stop=False))
    body = protein[:-1] if protein.endswith('*') else protein
    if '*' in body:
        oxa_frame_bad.append((header, 'internal_stop', body.index('*')))
        continue
    oxa_frame_ok.append(header)

print(f'{len(oxa_frame_ok)} / {len(oxa_frame_ok) + len(oxa_frame_bad)} sequences passed frame-integrity checks')
if oxa_frame_bad:
    print('Failures:', oxa_frame_bad[:10])

# Write only the passing sequences forward for alignment.
oxa_cds_input_path = os.path.join(OXA_PILOT_DIR, 'oxa_cds_input.fasta')
oxa_frame_ok_set = set(oxa_frame_ok)
with open(oxa_cds_input_path, 'w') as out:
    for header, seq in read_fasta_records(oxa_pooled_nt_path):
        if header in oxa_frame_ok_set:
            out.write(f'>{header}\n{seq}\n')

print(f'{len(oxa_frame_ok)} sequences written to {oxa_cds_input_path} for alignment')

321 / 321 sequences passed frame-integrity checks
321 sequences written to /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Phylogenetics/OXA_CDS_Pilot/oxa_cds_input.fasta for alignment


## 18.3 MAFFT alignment (nucleotide/CDS)

In [79]:
import subprocess

oxa_cds_aligned_path = os.path.join(OXA_PILOT_DIR, 'oxa_cds_aligned.fasta')
mafft_log_path = os.path.join(OXA_PILOT_DIR, 'mafft.log')

with open(oxa_cds_aligned_path, 'w') as out, open(mafft_log_path, 'w') as log:
    result = subprocess.run(
        [MAFFT_BIN, '--auto', '--thread', '16', '--inputorder', oxa_cds_input_path],
        stdout=out, stderr=log, text=True,
    )
if result.returncode != 0:
    raise RuntimeError(f'MAFFT failed, see {mafft_log_path}')

oxa_aligned_records = read_fasta_records(oxa_cds_aligned_path)
alignment_lengths = {len(seq) for _, seq in oxa_aligned_records}
print(f'MAFFT alignment complete: {len(oxa_aligned_records)} sequences, alignment length {alignment_lengths}')
print(f'-> {oxa_cds_aligned_path}')

MAFFT alignment complete: 321 sequences, alignment length {1623}
-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Phylogenetics/OXA_CDS_Pilot/oxa_cds_aligned.fasta


## 19. Tree: OXA beta-lactamase (CDS alignment)

FastTree on the nucleotide/CDS alignment from section 18, using the GTR model (this project's standard nucleotide-tree choice, per the phylogenetics conventions used in the beta-lactamase pilot).

### 19.1 Build the tree with FastTree

In [80]:
oxa_cds_tree_path = os.path.join(OXA_PILOT_DIR, 'oxa_cds_tree.nwk')
fasttree_log_path = os.path.join(OXA_PILOT_DIR, 'fasttree.log')

with open(oxa_cds_tree_path, 'w') as out, open(fasttree_log_path, 'w') as log:
    result = subprocess.run(
        [FASTTREE_BIN, '-nt', '-gtr', oxa_cds_aligned_path],
        stdout=out, stderr=log, text=True,
    )
if result.returncode != 0:
    raise RuntimeError(f'FastTree failed, see {fasttree_log_path}')

print(f'FastTree complete -> {oxa_cds_tree_path}')

FastTree complete -> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Phylogenetics/OXA_CDS_Pilot/oxa_cds_tree.nwk


### 19.2 Verify the tree

In [81]:
from Bio import Phylo

oxa_tree = Phylo.read(oxa_cds_tree_path, 'newick')
oxa_tree_leaves = oxa_tree.get_terminals()
print(f'{len(oxa_tree_leaves)} leaves in the OXA CDS tree (expected {len(oxa_frame_ok)})')
print(f'Total branch length: {oxa_tree.total_branch_length():.3f}')

321 leaves in the OXA CDS tree (expected 321)
Total branch length: 7.216


## 20. Metadata for the all-beta-lactam CD-HIT collection

Same metadata build as section 17, but for the all-beta-lactam-family collection (57 families, relabeled and CD-HIT-deduplicated separately -- `All_Beta_Lactam_Relabeled/CD/`). Corrects the bug in the manually-added cells above: the extraction loop was iterating `gram_negative_family_short_names` (the original 30-family list, which includes non-beta-lactam families like `16S rRNA methyltransferase (G1405)` -- causing the `FileNotFoundError`) instead of `all_beta_lactam_family_short_names`, and had no handling for organisms missing a given family (AB/ENT/Kp/Pa have 20/34/29/27 of the 57 families respectively, not all 57).

Reuses `all_beta_lactam_family_short_names` and `DEDUP_ROOT`/`DEDUP_ORG_FOLDERS` (already defined above), plus `parse_relabeled_fasta_gene_lengths`, `genes_per_family_per_organism`, and `gene_length_per_family_per_organism` (sections 14 and 16) unchanged. Variables here are prefixed `BL_` to keep them distinct from the earlier (buggy) `AB_dedup_gene_length_df`-style cells rather than silently overwriting them.

### 20.1 Per-organism gene + length extraction

Same per-organism pattern as section 17.1, with a skip for families a given organism doesn't have.

In [91]:
# AB (beta-lactam CD-HIT collection): gene name + sequence length per representative record.
BL_AB_gene_length_rows = []
for family_name in all_beta_lactam_family_short_names:
    path = os.path.join(DEDUP_ROOT, DEDUP_ORG_FOLDERS['AB'], f'{family_name}.fasta')
    if not os.path.exists(path):
        continue  # this organism doesn't have this family -- skip, don't crash
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        BL_AB_gene_length_rows.append(
            {'Organism': 'AB', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

BL_AB_gene_length_df = pd.DataFrame(BL_AB_gene_length_rows)
print(f'AB: {len(BL_AB_gene_length_df)} records, {BL_AB_gene_length_df["Gene"].nunique()} distinct genes, '
      f'{BL_AB_gene_length_df["AMR Gene Family"].nunique()} families present')

AB: 402 records, 196 distinct genes, 20 families present


In [92]:
# ENT (beta-lactam CD-HIT collection): gene name + sequence length per representative record.
BL_ENT_gene_length_rows = []
for family_name in all_beta_lactam_family_short_names:
    path = os.path.join(DEDUP_ROOT, DEDUP_ORG_FOLDERS['ENT'], f'{family_name}.fasta')
    if not os.path.exists(path):
        continue  # this organism doesn't have this family -- skip, don't crash
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        BL_ENT_gene_length_rows.append(
            {'Organism': 'ENT', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

BL_ENT_gene_length_df = pd.DataFrame(BL_ENT_gene_length_rows)
print(f'ENT: {len(BL_ENT_gene_length_df)} records, {BL_ENT_gene_length_df["Gene"].nunique()} distinct genes, '
      f'{BL_ENT_gene_length_df["AMR Gene Family"].nunique()} families present')

ENT: 760 records, 186 distinct genes, 34 families present


In [93]:
# Kp (beta-lactam CD-HIT collection): gene name + sequence length per representative record.
BL_Kp_gene_length_rows = []
for family_name in all_beta_lactam_family_short_names:
    path = os.path.join(DEDUP_ROOT, DEDUP_ORG_FOLDERS['Kp'], f'{family_name}.fasta')
    if not os.path.exists(path):
        continue  # this organism doesn't have this family -- skip, don't crash
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        BL_Kp_gene_length_rows.append(
            {'Organism': 'Kp', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

BL_Kp_gene_length_df = pd.DataFrame(BL_Kp_gene_length_rows)
print(f'Kp: {len(BL_Kp_gene_length_df)} records, {BL_Kp_gene_length_df["Gene"].nunique()} distinct genes, '
      f'{BL_Kp_gene_length_df["AMR Gene Family"].nunique()} families present')

Kp: 1111 records, 245 distinct genes, 29 families present


In [94]:
# Pa (beta-lactam CD-HIT collection): gene name + sequence length per representative record.
BL_Pa_gene_length_rows = []
for family_name in all_beta_lactam_family_short_names:
    path = os.path.join(DEDUP_ROOT, DEDUP_ORG_FOLDERS['Pa'], f'{family_name}.fasta')
    if not os.path.exists(path):
        continue  # this organism doesn't have this family -- skip, don't crash
    for gene_name, length in parse_relabeled_fasta_gene_lengths(path):
        BL_Pa_gene_length_rows.append(
            {'Organism': 'Pa', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

BL_Pa_gene_length_df = pd.DataFrame(BL_Pa_gene_length_rows)
print(f'Pa: {len(BL_Pa_gene_length_df)} records, {BL_Pa_gene_length_df["Gene"].nunique()} distinct genes, '
      f'{BL_Pa_gene_length_df["AMR Gene Family"].nunique()} families present')

Pa: 661 records, 271 distinct genes, 27 families present


### 20.2 Combine across organisms

In [95]:
BL_gene_length_long_df = pd.concat(
    [BL_AB_gene_length_df, BL_ENT_gene_length_df, BL_Kp_gene_length_df, BL_Pa_gene_length_df],
    ignore_index=True,
)
print(f'{len(BL_gene_length_long_df)} total records across all 4 organisms')
BL_gene_length_long_df.head()

2934 total records across all 4 organisms


,Organism,AMR Gene Family,Gene,Length
0,AB,ADC beta-lactamase without carbapenemase activity,ADC-11,1152
1,AB,ADC beta-lactamase without carbapenemase activity,ADC-79,1152
2,AB,ADC beta-lactamase without carbapenemase activity,ADC-30,1152
3,AB,ADC beta-lactamase without carbapenemase activity,ADC-76,1161
4,AB,ADC beta-lactamase without carbapenemase activity,ADC-82,1152


In [101]:
# Save the shared-family list with per-organism detail.
BL_gene_length_long_df.to_csv(
    os.path.join(BASE_DIR, 'Results/Genefamily/All_Beta_Lactam_Relabeled/CDGene_length_distribution.csv')
)

### 20.3 Per-(Organism, Family, Gene) summary

Reuses `gene_length_per_family_per_organism` (section 16) unchanged.

In [96]:
BL_gene_length_per_organism_df = gene_length_per_family_per_organism(BL_gene_length_long_df)
BL_gene_length_per_organism_df

,Organism,AMR Gene Family,Gene,N_Records,Min_Length,Max_Length,N_Distinct_Lengths
275,ENT,ACC beta-lactamase,ACC-1,1,1173,1173,1
469,Kp,ACC beta-lactamase,ACC-1,1,1173,1173,1
276,ENT,ACT beta-lactamase,ACT-1,1,1146,1146,1
277,ENT,ACT beta-lactamase,ACT-101,1,1146,1146,1
278,ENT,ACT beta-lactamase,ACT-102,1,1146,1146,1
...,...,...,...,...,...,...,...
1079,Pa,VIM beta-lactamase,VIM-6,1,936,936,1
1080,Pa,VIM beta-lactamase,VIM-80,1,933,933,1
1081,Pa,VIM beta-lactamase,VIM-84,1,801,801,1
1082,Pa,VIM beta-lactamase,VIM-85,1,936,936,1


In [102]:
# Save the shared-family list with per-organism detail.
BL_gene_length_per_organism_df.to_csv(
    os.path.join(BASE_DIR, 'Results/Genefamily/All_Beta_Lactam_Relabeled/CD/Gene_length_distribution_2.csv')
)

### 20.4 Pooled per-gene summary across organisms

In [97]:
BL_gene_length_summary_df = (
    BL_gene_length_long_df
    .groupby(['AMR Gene Family', 'Gene'])
    .agg(
        Organisms_Present=('Organism', lambda s: ', '.join(sorted(s.unique()))),
        N_Organisms=('Organism', 'nunique'),
        N_Records=('Gene', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
        N_Distinct_Lengths=('Length', 'nunique'),
    )
    .reset_index()
    .sort_values(['AMR Gene Family', 'Gene'])
)
BL_gene_length_summary_df

,AMR Gene Family,Gene,Organisms_Present,N_Organisms,N_Records,Min_Length,Max_Length,N_Distinct_Lengths
0,ACC beta-lactamase,ACC-1,"ENT, Kp",2,2,1173,1173,1
1,ACT beta-lactamase,ACT-1,ENT,1,1,1146,1146,1
2,ACT beta-lactamase,ACT-101,ENT,1,1,1146,1146,1
3,ACT beta-lactamase,ACT-102,ENT,1,1,1146,1146,1
4,ACT beta-lactamase,ACT-104,ENT,1,1,1146,1146,1
...,...,...,...,...,...,...,...,...
967,VIM beta-lactamase,VIM-6,Pa,1,1,936,936,1
968,VIM beta-lactamase,VIM-80,Pa,1,1,933,933,1
969,VIM beta-lactamase,VIM-84,Pa,1,1,801,801,1
970,VIM beta-lactamase,VIM-85,Pa,1,1,936,936,1


### 20.5 Per-family gene count

In [98]:
BL_family_gene_count_df = (
    BL_gene_length_summary_df
    .groupby('AMR Gene Family')
    .agg(N_Distinct_Genes=('Gene', 'nunique'))
    .reset_index()
    .sort_values('N_Distinct_Genes', ascending=False)
)
BL_family_gene_count_df

,AMR Gene Family,N_Distinct_Genes
28,OXA beta-lactamase,173
42,PDC beta-lactamase,104
1,ACT beta-lactamase,85
3,ADC beta-lactamases pending classification for...,74
50,SHV beta-lactamase,68
39,OXA-51-like beta-lactamase,63
38,OXA-50-like beta-lactamase,58
21,KPC beta-lactamase,57
9,CTX-M beta-lactamase,32
53,TEM beta-lactamase,27


### 20.6 Per-organism gene-count matrix

Reuses `genes_per_family_per_organism` (section 16) unchanged.

In [99]:
BL_genes_per_family_per_organism_df = genes_per_family_per_organism(BL_gene_length_long_df)
BL_genes_per_family_per_organism_df

Organism,AB,ENT,Kp,Pa
AMR Gene Family,,,,
ACC beta-lactamase,0,1,1,0
ACT beta-lactamase,0,85,0,0
ADC beta-lactamase without carbapenemase activity,19,0,0,0
ADC beta-lactamases pending classification for carbapenemase activity,74,0,0,0
AFM beta-lactamase,0,0,0,3
BKC Beta-lactamase,0,1,0,0
CARB beta-lactamase,3,1,1,3
CMH beta-lactamase,0,14,0,0
CMY beta-lactamase,1,2,9,1


### 20.7 Per-organism, per-family length summary (no gene breakdown)

In [100]:
BL_organism_family_length_summary_df = (
    BL_gene_length_long_df
    .groupby(['Organism', 'AMR Gene Family'])
    .agg(
        N_Records=('Length', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
    )
    .reset_index()
    .sort_values(['AMR Gene Family', 'Organism'])
)
BL_organism_family_length_summary_df

,Organism,AMR Gene Family,N_Records,Min_Length,Max_Length
20,ENT,ACC beta-lactamase,1,1173,1173
54,Kp,ACC beta-lactamase,1,1173,1173
21,ENT,ACT beta-lactamase,163,1005,1248
0,AB,ADC beta-lactamase without carbapenemase activity,35,1152,1167
1,AB,ADC beta-lactamases pending classification for...,105,960,1170
...,...,...,...,...,...
108,Pa,VEB beta-lactamase,9,789,900
53,ENT,VIM beta-lactamase,2,801,801
82,Kp,VIM beta-lactamase,3,801,801
109,Pa,VIM beta-lactamase,12,801,936


In [103]:
# Save the shared-family list with per-organism detail.
BL_organism_family_length_summary_df.to_csv(
    os.path.join(BASE_DIR, 'Results/Genefamily/All_Beta_Lactam_Relabeled/CD/Gene_length_distribution_3.csv')
)

## 21. Pool all beta-lactam sequences (CDS)

Modified from section 18.1's OXA-only pooling: instead of one hardcoded family (`OXA_FAMILY`, `.fasta.fasta` extension matching the old `Dedup_03_08_26` pilot), this loops over all 57 families in `all_beta_lactam_family_short_names`, reads from the all-beta-lactam CD-HIT collection (`DEDUP_ROOT`/`DEDUP_ORG_FOLDERS`, single `.fasta` extension), and skips a family an organism doesn't have rather than assuming every organism has every family.

In [106]:
BETA_LACTAM_POOL_DIR = os.path.join(BASE_DIR, 'Results/Phylogenetics/Beta_lactamase_all')
os.makedirs(BETA_LACTAM_POOL_DIR, exist_ok=True)

beta_lactam_pooled_nt_path = os.path.join(BETA_LACTAM_POOL_DIR, 'beta_lactam_pooled_nt.fasta')
beta_lactam_pooled_counts = {}
with open(beta_lactam_pooled_nt_path, 'w') as out:
    for org in GN_RELABEL_ORGS:
        org_count = 0
        for family_name in all_beta_lactam_family_short_names:
            path = os.path.join(DEDUP_ROOT, DEDUP_ORG_FOLDERS[org], f'{family_name}.fasta')
            if not os.path.exists(path):
                continue  # this organism doesn't have this family -- skip, don't crash
            records = read_fasta_records(path)
            for header, seq in records:
                out.write(f'>{header}\n{seq}\n')
            org_count += len(records)
        beta_lactam_pooled_counts[org] = org_count

print('Per-organism record counts:', beta_lactam_pooled_counts)
print('Total pooled:', sum(beta_lactam_pooled_counts.values()))

Per-organism record counts: {'AB': 402, 'ENT': 760, 'Kp': 1111, 'Pa': 661}
Total pooled: 2934


In [107]:
from Bio.Seq import Seq

beta_lactam_frame_ok = []
beta_lactam_frame_bad = []
for header, seq in read_fasta_records(beta_lactam_pooled_nt_path):
    seq = seq.upper()
    if len(seq) % 3 != 0:
        beta_lactam_frame_bad.append((header, 'bad_length', len(seq)))
        continue
    protein = str(Seq(seq).translate(table=11, to_stop=False))
    body = protein[:-1] if protein.endswith('*') else protein
    if '*' in body:
        beta_lactam_frame_bad.append((header, 'internal_stop', body.index('*')))
        continue
    beta_lactam_frame_ok.append(header)

print(f'{len(beta_lactam_frame_ok)} / {len(beta_lactam_frame_ok) + len(beta_lactam_frame_bad)} sequences passed frame-integrity checks')
if beta_lactam_frame_bad:
    print('Failures:', beta_lactam_frame_bad[:10])

# Write only the passing sequences forward for alignment.
beta_lactam_cds_input_path = os.path.join(BETA_LACTAM_POOL_DIR, 'betalactam_cds_input.fasta')
beta_lactam_frame_ok_set = set(beta_lactam_frame_ok)
with open(beta_lactam_cds_input_path, 'w') as out:
    for header, seq in read_fasta_records(beta_lactam_pooled_nt_path):
        if header in beta_lactam_frame_ok_set:
            out.write(f'>{header}\n{seq}\n')

print(f'{len(beta_lactam_frame_ok)} sequences written to {beta_lactam_cds_input_path} for alignment')

2934 / 2934 sequences passed frame-integrity checks
2934 sequences written to /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Phylogenetics/Beta_lactamase_all/betalactam_cds_input.fasta for alignment


In [109]:
import subprocess

betalactam_cds_aligned_path = os.path.join(BETA_LACTAM_POOL_DIR, 'betalactam_cds_aligned.fasta')
mafft_log_path = os.path.join(BETA_LACTAM_POOL_DIR, 'mafft.log')

with open(betalactam_cds_aligned_path, 'w') as out, open(mafft_log_path, 'w') as log:
    result = subprocess.run(
        [MAFFT_BIN, '--auto', '--thread', '16', '--inputorder', beta_lactam_cds_input_path],
        stdout=out, stderr=log, text=True,
    )
if result.returncode != 0:
    raise RuntimeError(f'MAFFT failed, see {mafft_log_path}')

beta_lactam_aligned_records = read_fasta_records(betalactam_cds_aligned_path)
alignment_lengths = {len(seq) for _, seq in beta_lactam_aligned_records}
print(f'MAFFT alignment complete: {len(beta_lactam_aligned_records)} sequences, alignment length {alignment_lengths}')
print(f'-> {betalactam_cds_aligned_path}')

MAFFT alignment complete: 2934 sequences, alignment length {4726}
-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Phylogenetics/Beta_lactamase_all/betalactam_cds_aligned.fasta


In [111]:
# Sanitize headers before tree-building -- see root-cause note above.
betalactam_cds_aligned_safe_path = os.path.join(BETA_LACTAM_POOL_DIR, 'betalactam_cds_aligned_input.fasta')
with open(betalactam_cds_aligned_safe_path, 'w') as out:
    for header, seq in read_fasta_records(betalactam_cds_aligned_path):
        safe_header = header.replace(' ', '_')
        out.write(f'>{safe_header}\n{seq}\n')

print(f'Sanitized headers -> {betalactam_cds_aligned_safe_path}')

# Build the tree from the sanitized alignment.
cds_tree_path = os.path.join(BETA_LACTAM_POOL_DIR, 'cds_tree.nwk')
fasttree_log_path = os.path.join(BETA_LACTAM_POOL_DIR, 'fasttree.log')

with open(cds_tree_path, 'w') as out, open(fasttree_log_path, 'w') as log:
    result = subprocess.run(
        [FASTTREE_BIN, '-nt', '-gtr', betalactam_cds_aligned_safe_path],
        stdout=out, stderr=log, text=True,
    )
if result.returncode != 0:
    raise RuntimeError(f'FastTree failed, see {fasttree_log_path}')

print(f'FastTree complete -> {cds_tree_path}')

Sanitized headers -> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Phylogenetics/Beta_lactamase_all/betalactam_cds_aligned_input.fasta
FastTree complete -> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Phylogenetics/Beta_lactamase_all/cds_tree.nwk


### Redo work

In [118]:
import pandas as pd
df_amr_dist = pd.read_csv("/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Counts_percentage.csv")
df_amr_dist.head(5).style


,Organism,AMR Gene Family,Count,Unnamed: 3,Unnamed: 4
0,Acinetobacter baumannii,APH(4),1,nan,nan
1,Acinetobacter baumannii,resistance-nodulation-cell division (RND) antibiotic efflux pump,14,nan,nan
2,Acinetobacter baumannii,fluoroquinolone resistant parC,1,nan,nan
3,Acinetobacter baumannii,small multidrug resistance (SMR) antibiotic efflux pump,3,nan,nan
4,Acinetobacter baumannii,GIM beta-lactamase,1,nan,nan


In [123]:
# Pass the column names as a single list to the 'columns' parameter
df1 = df_amr_dist.head(5).drop(columns=['Unnamed: 3', 'Unnamed: 4']).style
df1

,Organism,AMR Gene Family,Count
0,Acinetobacter baumannii,APH(4),1
1,Acinetobacter baumannii,resistance-nodulation-cell division (RND) antibiotic efflux pump,14
2,Acinetobacter baumannii,fluoroquinolone resistant parC,1
3,Acinetobacter baumannii,small multidrug resistance (SMR) antibiotic efflux pump,3
4,Acinetobacter baumannii,GIM beta-lactamase,1


In [124]:
import pandas as pd

# Load the dataset
df_amr_dist = pd.read_csv("/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Counts_percentage.csv")

# Extract and sort unique families
unique_families = sorted(df_amr_dist['AMR Gene Family'].dropna().unique())

# Create a mapping dictionary generating IDs like 'f1', 'f2', ..., 'f141'
# We no longer need the 99-limit check, so it has been removed.
family_to_id = {family: f'f{i}' for i, family in enumerate(unique_families, start=1)}

# Find the position of 'AMR Gene Family' and insert the new ID column right after it
insert_pos = df_amr_dist.columns.get_loc('AMR Gene Family') + 1
df_amr_dist.insert(insert_pos, 'AMR_Gene_Family_ID', df_amr_dist['AMR Gene Family'].map(family_to_id))

# Validation step: Ensure every unique family maps to exactly one ID
check = df_amr_dist.groupby('AMR Gene Family')['AMR_Gene_Family_ID'].nunique()
assert (check == 1).all(), 'A family maps to more than one ID — check for stray whitespace in the source column.'

# Display the first 10 rows to verify the new 'f1', 'f2' format
df_amr_dist.head(10)

,Organism,AMR Gene Family,AMR_Gene_Family_ID,Count,Unnamed: 3,Unnamed: 4
0,Acinetobacter baumannii,APH(4),f20,1,NaN,NaN
1,Acinetobacter baumannii,resistance-nodulation-cell division (RND) anti...,f120,14,NaN,NaN
2,Acinetobacter baumannii,fluoroquinolone resistant parC,f105,1,NaN,NaN
3,Acinetobacter baumannii,small multidrug resistance (SMR) antibiotic ef...,f123,3,NaN,NaN
4,Acinetobacter baumannii,GIM beta-lactamase,f38,1,NaN,NaN
5,Acinetobacter baumannii,ADC beta-lactamases pending classification for...,f10,77,NaN,NaN
6,Acinetobacter baumannii,macrolide phosphotransferase (MPH),f113,1,NaN,NaN
7,Acinetobacter baumannii,AAC(6'),f5,4,NaN,NaN
8,Acinetobacter baumannii,OXA-51-like beta-lactamase,f65,63,NaN,NaN
9,Acinetobacter baumannii,SHV beta-lactamase,f77,2,NaN,NaN


In [125]:
# Save the shared-family list with per-organism detail.
df_amr_dist.to_csv(
    os.path.join(BASE_DIR, '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_unique.csv')
)

In [128]:
import pandas as pd

# Assuming your dataframe is already loaded, for example:
df_amr_dist = pd.read_csv("/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/AMR_GeneDistribution/ESKAPE_AMR_Gene_Family_Counts_percentage.csv")

# 1. Select the column, drop empty values, and get unique entries
unique_values_array = df_amr_dist['AMR Gene Family'].dropna().unique()

# 2. (Optional) Convert the numpy array to a standard Python list
unique_values_list = unique_values_array.tolist()

# 3. Print the result to check it out
print(f"Found {len(unique_values_list)} unique numbers/IDs.")
print(unique_values_list)

Found 141 unique numbers/IDs.
['APH(4)', 'resistance-nodulation-cell division (RND) antibiotic efflux pump', 'fluoroquinolone resistant parC', 'small multidrug resistance (SMR) antibiotic efflux pump', 'GIM beta-lactamase', 'ADC beta-lactamases pending classification for carbapenemase activity', 'macrolide phosphotransferase (MPH)', "AAC(6')", 'OXA-51-like beta-lactamase', 'SHV beta-lactamase', 'tetracycline-resistant ribosomal protection protein', '16S rRNA methyltransferase (G1405)', 'ADC beta-lactamase without carbapenemase activity', 'sulfonamide resistant sul', 'rifampin ADP-ribosyltransferase (Arr)', 'OXA-24-like beta-lactamase', 'TEM beta-lactamase', 'macrolide esterase', "ANT(2'')", 'MCR phosphoethanolamine transferase', 'fosfomycin thiol transferase', 'OXA beta-lactamase', 'AAC(3)', 'APH(6)', 'GES beta-lactamase', "ANT(3'')", 'fluoroquinolone resistant gyrA', 'OXA-58-like beta-lactamase', 'IMP beta-lactamase', 'OXA-2-like beta-lactamase', 'trimethoprim resistant dihydrofolate 

In [129]:
import pandas as pd
import os

# 1. Convert our existing 'family_to_id' dictionary into a new DataFrame
# .items() pulls out the pairs (gene family, f-code) so pandas can structure them as rows
mapping_df = pd.DataFrame(list(family_to_id.items()), columns=['AMR_FAMILY_UNIQUE_NAMES', 'ID'])

# 2. Define exactly where you want to save this new file
# Adjust this path if you want it saved somewhere else!
output_path = "/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/AMR_GeneDistribution/AMR_Family_ID_Mapping.csv"

# 3. Save the dataframe as a CSV
# index=False ensures we only keep the two columns we care about
mapping_df.to_csv(output_path, index=False)

# Print a success message and show the first 5 rows to confirm it looks perfect
print(f"Successfully saved unique ID mapping to: {output_path}")
mapping_df.head(5).style

Successfully saved unique ID mapping to: /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/AMR_GeneDistribution/AMR_Family_ID_Mapping.csv


,AMR_FAMILY_UNIQUE_NAMES,ID
0,16S rRNA methyltransferase (G1405),f1
1,23S rRNA with mutation conferring resistance to linezolid antibiotics,f2
2,AAC(2'),f3
3,AAC(3),f4
4,AAC(6'),f5


## 22. Rename ALL family headers using `AMR_Family_ID_Mapping.csv`

New universal header format: `f<ID>_<organism>_<gene_name>_<index>`, ID from `AMR_Family_ID_Mapping.csv` (covers all 141 families, not just beta-lactam -- the later sections filter down to the beta-lactam subset). Two deliberate additions beyond the literal 3-part spec, both directly motivated by problems hit earlier in this notebook:

- **Trailing index** -- without it, every repeated occurrence of the same gene name within one (organism, family) file collides on an identical header (this is exactly the problem the original relabeling scheme in section 14 was built to solve -- e.g. AB's `armA` alone repeats 424 times).
- **Spaces in the gene name replaced with underscores at relabel time** -- some CARD gene names are full free-text descriptions (e.g. `Escherichia coli soxS with mutation conferring antibiotic resistance`), which is exactly what caused the FastTree "Non-unique name" crash a few sections ago. Sanitizing here, once, up front, means every later section built on this new scheme never needs the separate post-hoc sanitization step that was needed before.

**Per instruction: code only, not executed -- no files written.**

In [131]:
import csv

AMR_FAMILY_ID_MAPPING_PATH = os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/AMR_Family_ID_Mapping.csv')

family_name_to_id = {}
with open(AMR_FAMILY_ID_MAPPING_PATH) as f:
    reader = csv.DictReader(f)
    for row in reader:
        family_name_to_id[row['AMR_FAMILY_UNIQUE_NAMES']] = row['ID']

print(f'{len(family_name_to_id)} families loaded from the ID mapping file')

142 families loaded from the ID mapping file


In [132]:
def relabel_fasta_headers_by_id(input_path, output_path, org_code, family_id):
    """
    Rewrite one gene-family FASTA's headers as 'f<ID>_<org>_<gene>_<i>' (1-based,
    per-file counter) -- family ID from AMR_Family_ID_Mapping.csv, gene name with
    internal whitespace replaced by underscores, trailing counter for uniqueness.
    Returns the number of records written.
    """
    records = []
    gene_name = None
    seq_lines = []
    with open(input_path) as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('>'):
                if gene_name is not None:
                    records.append((gene_name, seq_lines))
                gene_name = line[1:].strip()
                seq_lines = []
            else:
                seq_lines.append(line)
        if gene_name is not None:
            records.append((gene_name, seq_lines))

    with open(output_path, 'w') as out:
        for i, (gene, seq) in enumerate(records, start=1):
            safe_gene = gene.replace(' ', '_')
            out.write(f'>{family_id}_{org_code}_{safe_gene}_{i}\n')
            for seq_line in seq:
                out.write(seq_line + '\n')

    return len(records)

### 22.1 Relabel every family, every organism

Same per-organism structure as section 14.2, looping the full 141-family ID mapping this time and skipping a family an organism doesn't have.

In [138]:
ID_RELABEL_OUTPUT_ROOT = os.path.join(GENEFAMILY_DIR, 'All_Family_ID_Relabeled_2')
ID_RELABEL_ORG_FOLDERS = {
    'AB':  'AB_ARG_Fasta_Files',
    'ENT': 'ENT_ARG_Fasta_Files',
    'Kp':  'Kp_ARG_Fasta_Files',
    'Pa':  'Pa_ARG_Fasta_Files',
    'EF':  'EF_ARG_Fasta_Files',
    'SA':  'SA_ARG_Fasta_Files',
}

for org, folder in ID_RELABEL_ORG_FOLDERS.items():
    input_dir = os.path.join(GENEFAMILY_DIR, folder)
    output_dir = os.path.join(ID_RELABEL_OUTPUT_ROOT, org)
    os.makedirs(output_dir, exist_ok=True)

    org_counts = {}
    for family_name, family_id in family_name_to_id.items():
        input_path = os.path.join(input_dir, f'{family_name}.fasta')
        if not os.path.exists(input_path):
            continue  # this organism doesn't have this family -- skip, don't crash
        output_path = os.path.join(output_dir, f'{family_name}.fasta')
        org_counts[family_name] = relabel_fasta_headers_by_id(input_path, output_path, org, family_id)

    print(f'{org}: {len(org_counts)} families relabeled, {sum(org_counts.values())} total records')

AB: 50 families relabeled, 33629 total records
ENT: 73 families relabeled, 29168 total records
Kp: 77 families relabeled, 189098 total records
Pa: 63 families relabeled, 94360 total records
EF: 32 families relabeled, 17744 total records
SA: 50 families relabeled, 52847 total records


**Next step (outside the notebook, same as before):** run CD-HIT-EST at 100% identity on `All_Family_ID_Relabeled/<org>/` (or just the beta-lactam-family subset of it) once the cell above has actually been run, the same way it was done for `All_Beta_Lactam_Relabeled/`. Sections 23-24 below assume that output lands in `All_Family_ID_Relabeled/CD/<org>/` -- adjust `ID_DEDUP_ROOT`/`ID_DEDUP_ORG_FOLDERS` in section 23.1 if the real output folder names differ (check casing, e.g. `KP`/`PA` vs `Kp`/`Pa`, the same gotcha hit twice already).

## 23. Metadata for the beta-lactam subset (ID-relabeled scheme)

Same metadata build as sections 17/20, filtered to the 57 beta-lactam families (reusing `all_beta_lactam_family_short_names`'s keys purely as the family-name filter list -- the short codes themselves aren't used here, only the ID scheme is), reading from the new `f<ID>_<org>_<gene>_<i>` header format instead.

In [139]:
def parse_id_relabeled_fasta_gene_lengths(path):
    """
    Parse an ID-relabeled FASTA file ('f<ID>_<org>_<gene>_<i>' headers), recovering the
    gene name (spaces restored) and sequence length per record. Family ID and organism
    are always the first two underscore-separated tokens and the index is always the
    last, so the gene name in between -- underscore-joined wherever it used to contain
    spaces -- is recovered by dropping those and converting underscores back to spaces.
    Returns a list of (gene_name, length) tuples, matching
    parse_relabeled_fasta_gene_lengths's signature so the section 16/17 summary
    functions can be reused unchanged.
    """
    records = []
    header = None
    seq_len = 0
    with open(path) as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('>'):
                if header is not None:
                    records.append((header, seq_len))
                header = line[1:].strip()
                seq_len = 0
            else:
                seq_len += len(line)
        if header is not None:
            records.append((header, seq_len))

    return [
        ('_'.join(header.split('_')[2:-1]).replace('_', ' '), length)
        for header, length in records
    ]

### 23.1 Config

Pending the external CD-HIT run noted above -- update the folder names here once real output exists.

In [140]:
ID_DEDUP_ROOT = os.path.join(ID_RELABEL_OUTPUT_ROOT, 'CD')
ID_DEDUP_ORG_FOLDERS = {
    'AB':  'AB Representative sequences',
    'ENT': 'ENT Representative sequences',
    'Kp':  'Kp Representative sequences',
    'Pa':  'pa Representative sequences',
    'EF':  'EF Representative sequences',
    'SA':  'SA  Representative sequences'
}

### 23.2 Per-organism gene + length extraction (beta-lactam subset)

In [141]:
# AB (ID-relabeled, beta-lactam subset): gene name + sequence length per representative record.
ID_AB_gene_length_rows = []
for family_name in all_beta_lactam_family_short_names:
    family_id = family_name_to_id[family_name]
    path = os.path.join(ID_DEDUP_ROOT, ID_DEDUP_ORG_FOLDERS['AB'], f'{family_name}.fasta')
    if not os.path.exists(path):
        continue  # this organism doesn't have this family -- skip, don't crash
    for gene_name, length in parse_id_relabeled_fasta_gene_lengths(path):
        ID_AB_gene_length_rows.append(
            {'Organism': 'AB', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

ID_AB_gene_length_df = pd.DataFrame(ID_AB_gene_length_rows)
print(f'AB: {len(ID_AB_gene_length_df)} records')

AB: 402 records


In [142]:
# ENT (ID-relabeled, beta-lactam subset): gene name + sequence length per representative record.
ID_ENT_gene_length_rows = []
for family_name in all_beta_lactam_family_short_names:
    family_id = family_name_to_id[family_name]
    path = os.path.join(ID_DEDUP_ROOT, ID_DEDUP_ORG_FOLDERS['ENT'], f'{family_name}.fasta')
    if not os.path.exists(path):
        continue  # this organism doesn't have this family -- skip, don't crash
    for gene_name, length in parse_id_relabeled_fasta_gene_lengths(path):
        ID_ENT_gene_length_rows.append(
            {'Organism': 'ENT', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

ID_ENT_gene_length_df = pd.DataFrame(ID_ENT_gene_length_rows)
print(f'ENT: {len(ID_ENT_gene_length_df)} records')

ENT: 760 records


In [143]:
# Kp (ID-relabeled, beta-lactam subset): gene name + sequence length per representative record.
ID_Kp_gene_length_rows = []
for family_name in all_beta_lactam_family_short_names:
    family_id = family_name_to_id[family_name]
    path = os.path.join(ID_DEDUP_ROOT, ID_DEDUP_ORG_FOLDERS['Kp'], f'{family_name}.fasta')
    if not os.path.exists(path):
        continue  # this organism doesn't have this family -- skip, don't crash
    for gene_name, length in parse_id_relabeled_fasta_gene_lengths(path):
        ID_Kp_gene_length_rows.append(
            {'Organism': 'Kp', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

ID_Kp_gene_length_df = pd.DataFrame(ID_Kp_gene_length_rows)
print(f'Kp: {len(ID_Kp_gene_length_df)} records')

Kp: 1111 records


In [144]:
# Pa (ID-relabeled, beta-lactam subset): gene name + sequence length per representative record.
ID_Pa_gene_length_rows = []
for family_name in all_beta_lactam_family_short_names:
    family_id = family_name_to_id[family_name]
    path = os.path.join(ID_DEDUP_ROOT, ID_DEDUP_ORG_FOLDERS['Pa'], f'{family_name}.fasta')
    if not os.path.exists(path):
        continue  # this organism doesn't have this family -- skip, don't crash
    for gene_name, length in parse_id_relabeled_fasta_gene_lengths(path):
        ID_Pa_gene_length_rows.append(
            {'Organism': 'Pa', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

ID_Pa_gene_length_df = pd.DataFrame(ID_Pa_gene_length_rows)
print(f'Pa: {len(ID_Pa_gene_length_df)} records')

Pa: 661 records


## ENT & KP 

In [144]:
# ENT (ID-relabeled, beta-lactam subset): gene name + sequence length per representative record.
ID_ENT_gene_length_rows = []
for family_name in all_beta_lactam_family_short_names:
    family_id = family_name_to_id[family_name]
    path = os.path.join(ID_DEDUP_ROOT, ID_DEDUP_ORG_FOLDERS['Pa'], f'{family_name}.fasta')
    if not os.path.exists(path):
        continue  # this organism doesn't have this family -- skip, don't crash
    for gene_name, length in parse_id_relabeled_fasta_gene_lengths(path):
        ID_Pa_gene_length_rows.append(
            {'Organism': 'Pa', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

ID_Pa_gene_length_df = pd.DataFrame(ID_Pa_gene_length_rows)
print(f'Pa: {len(ID_Pa_gene_length_df)} records')

Pa: 661 records


In [144]:
# Pa (ID-relabeled, beta-lactam subset): gene name + sequence length per representative record.
ID_Pa_gene_length_rows = []
for family_name in all_beta_lactam_family_short_names:
    family_id = family_name_to_id[family_name]
    path = os.path.join(ID_DEDUP_ROOT, ID_DEDUP_ORG_FOLDERS['Pa'], f'{family_name}.fasta')
    if not os.path.exists(path):
        continue  # this organism doesn't have this family -- skip, don't crash
    for gene_name, length in parse_id_relabeled_fasta_gene_lengths(path):
        ID_Pa_gene_length_rows.append(
            {'Organism': 'Pa', 'AMR Gene Family': family_name, 'Gene': gene_name, 'Length': length}
        )

ID_Pa_gene_length_df = pd.DataFrame(ID_Pa_gene_length_rows)
print(f'Pa: {len(ID_Pa_gene_length_df)} records')

Pa: 661 records


### 23.3 Combine across organisms

In [ ]:
ID_gene_length_long_df = pd.concat(
    [ID_AB_gene_length_df, ID_ENT_gene_length_df, ID_Kp_gene_length_df, ID_Pa_gene_length_df],
    ignore_index=True,
)
print(f'{len(ID_gene_length_long_df)} total records across all 4 organisms')
ID_gene_length_long_df.head()

### 23.4 Per-(Organism, Family, Gene) summary

Reuses `gene_length_per_family_per_organism` (section 16) unchanged.

In [ ]:
ID_gene_length_per_organism_df = gene_length_per_family_per_organism(ID_gene_length_long_df)
ID_gene_length_per_organism_df

### 23.5 Per-organism, per-family length summary (min-max, no gene breakdown)

In [ ]:
ID_organism_family_length_summary_df = (
    ID_gene_length_long_df
    .groupby(['Organism', 'AMR Gene Family'])
    .agg(
        N_Records=('Length', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
    )
    .reset_index()
    .sort_values(['AMR Gene Family', 'Organism'])
)
ID_organism_family_length_summary_df

### 23.6 Per-family gene count

In [ ]:
ID_family_gene_count_df = (
    ID_gene_length_long_df
    .groupby('AMR Gene Family')['Gene']
    .nunique()
    .reset_index(name='N_Distinct_Genes')
    .sort_values('N_Distinct_Genes', ascending=False)
)
ID_family_gene_count_df

## 24. Pool, align, and build the tree for all beta-lactam CDS (ID-relabeled scheme)

Same as sections 21 + the MAFFT/FastTree cells fixed earlier, reading from the new ID-relabeled/deduplicated data. No separate header-sanitization step is needed this time -- gene names were already made whitespace-free at relabel time (section 22), so the FastTree "Non-unique name" bug from before cannot recur here.

### 24.1 Pool all beta-lactam sequences (CDS)

In [ ]:
ID_BETA_LACTAM_POOL_DIR = os.path.join(BASE_DIR, 'Results/Phylogenetics/Beta_Lactam_ID_Pool')
os.makedirs(ID_BETA_LACTAM_POOL_DIR, exist_ok=True)

id_beta_lactam_pooled_nt_path = os.path.join(ID_BETA_LACTAM_POOL_DIR, 'beta_lactam_pooled_nt.fasta')
id_beta_lactam_pooled_counts = {}
with open(id_beta_lactam_pooled_nt_path, 'w') as out:
    for org in GN_RELABEL_ORGS:
        org_count = 0
        for family_name in all_beta_lactam_family_short_names:
            path = os.path.join(ID_DEDUP_ROOT, ID_DEDUP_ORG_FOLDERS[org], f'{family_name}.fasta')
            if not os.path.exists(path):
                continue  # this organism doesn't have this family -- skip, don't crash
            records = read_fasta_records(path)
            for header, seq in records:
                out.write(f'>{header}\n{seq}\n')
            org_count += len(records)
        id_beta_lactam_pooled_counts[org] = org_count

print('Per-organism record counts:', id_beta_lactam_pooled_counts)
print('Total pooled:', sum(id_beta_lactam_pooled_counts.values()))

### 24.2 Frame-integrity check

In [ ]:
from Bio.Seq import Seq

id_beta_lactam_frame_ok = []
id_beta_lactam_frame_bad = []
for header, seq in read_fasta_records(id_beta_lactam_pooled_nt_path):
    seq = seq.upper()
    if len(seq) % 3 != 0:
        id_beta_lactam_frame_bad.append((header, 'bad_length', len(seq)))
        continue
    protein = str(Seq(seq).translate(table=11, to_stop=False))
    body = protein[:-1] if protein.endswith('*') else protein
    if '*' in body:
        id_beta_lactam_frame_bad.append((header, 'internal_stop', body.index('*')))
        continue
    id_beta_lactam_frame_ok.append(header)

print(f'{len(id_beta_lactam_frame_ok)} / {len(id_beta_lactam_frame_ok) + len(id_beta_lactam_frame_bad)} sequences passed frame-integrity checks')
if id_beta_lactam_frame_bad:
    print('Failures:', id_beta_lactam_frame_bad[:10])

id_beta_lactam_cds_input_path = os.path.join(ID_BETA_LACTAM_POOL_DIR, 'beta_lactam_cds_input.fasta')
id_beta_lactam_frame_ok_set = set(id_beta_lactam_frame_ok)
with open(id_beta_lactam_cds_input_path, 'w') as out:
    for header, seq in read_fasta_records(id_beta_lactam_pooled_nt_path):
        if header in id_beta_lactam_frame_ok_set:
            out.write(f'>{header}\n{seq}\n')

print(f'{len(id_beta_lactam_frame_ok)} sequences written to {id_beta_lactam_cds_input_path} for alignment')

### 24.3 MAFFT alignment (nucleotide/CDS)

In [ ]:
id_beta_lactam_cds_aligned_path = os.path.join(ID_BETA_LACTAM_POOL_DIR, 'beta_lactam_cds_aligned.fasta')
id_mafft_log_path = os.path.join(ID_BETA_LACTAM_POOL_DIR, 'mafft.log')

with open(id_beta_lactam_cds_aligned_path, 'w') as out, open(id_mafft_log_path, 'w') as log:
    result = subprocess.run(
        [MAFFT_BIN, '--auto', '--thread', '16', '--inputorder', id_beta_lactam_cds_input_path],
        stdout=out, stderr=log, text=True,
    )
if result.returncode != 0:
    raise RuntimeError(f'MAFFT failed, see {id_mafft_log_path}')

id_beta_lactam_aligned_records = read_fasta_records(id_beta_lactam_cds_aligned_path)
id_alignment_lengths = {len(seq) for _, seq in id_beta_lactam_aligned_records}
print(f'MAFFT alignment complete: {len(id_beta_lactam_aligned_records)} sequences, alignment length {id_alignment_lengths}')
print(f'-> {id_beta_lactam_cds_aligned_path}')

### 24.4 Build the tree with FastTree

No header-sanitization step needed here -- see section 22's note.

In [ ]:
id_cds_tree_path = os.path.join(ID_BETA_LACTAM_POOL_DIR, 'cds_tree.nwk')
id_fasttree_log_path = os.path.join(ID_BETA_LACTAM_POOL_DIR, 'fasttree.log')

with open(id_cds_tree_path, 'w') as out, open(id_fasttree_log_path, 'w') as log:
    result = subprocess.run(
        [FASTTREE_BIN, '-nt', '-gtr', id_beta_lactam_cds_aligned_path],
        stdout=out, stderr=log, text=True,
    )
if result.returncode != 0:
    raise RuntimeError(f'FastTree failed, see {id_fasttree_log_path}')

print(f'FastTree complete -> {id_cds_tree_path}')

## Header Renaming AND Metadata creation

In [ ]:
import csv, re, os

BASE_DIR = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family'
BETA_LACTAM_POOL_DIR = os.path.join(BASE_DIR, 'Results/Phylogenetics/Beta_lactamase_all')
MAPPING_CSV = os.path.join(BASE_DIR, 'Results/AMR_GeneDistribution/AMR_Family_ID_Mapping.csv')
INPUT_FASTA = os.path.join(BETA_LACTAM_POOL_DIR, 'betalactam_cds_input.fasta')
OUTPUT_FASTA = os.path.join(BETA_LACTAM_POOL_DIR, 'betalactam_cds_relabelled.fasta')
OUTPUT_LOOKUP = os.path.join(BETA_LACTAM_POOL_DIR, 'header_relabel_lookup.csv')

# full CARD family name -> short code, from your notebook's section 13 (cell 71)
all_beta_lactam_family_short_names = {
    'ACC beta-lactamase': 'ACC', 'ACT beta-lactamase': 'ACT',
    'ADC beta-lactamase without carbapenemase activity': 'ADCn',
    'ADC beta-lactamases pending classification for carbapenemase activity': 'ADCp',
    'AFM beta-lactamase': 'AFM', 'BKC Beta-lactamase': 'BKC', 'CARB beta-lactamase': 'CARB',
    'CMH beta-lactamase': 'CMH', 'CMY beta-lactamase': 'CMY', 'CTX-M beta-lactamase': 'CTXM',
    'DHA beta-lactamase': 'DHA', 'DIM beta-lactamase': 'DIM', 'FIM beta-lactamase': 'FIM',
    'FONA beta-lactamase': 'FONA', 'FRI beta-lactamase': 'FRI', 'GES beta-lactamase': 'GES',
    'GIM beta-lactamase': 'GIM',
    'General Bacterial Porin with reduced permeability to beta-lactams': 'PORIN',
    'IMI beta-lactamase': 'IMI', 'IMP beta-lactamase': 'IMP', 'KHM beta-lactamase': 'KHM',
    'KPC beta-lactamase': 'KPC', 'LAP beta-lactamase': 'LAP', 'LCR beta-lactamase': 'LCR',
    'LEN beta-lactamase': 'LEN', 'MIR beta-lactamase': 'MIR', 'MOX beta-lactamase': 'MOX',
    'NDM beta-lactamase': 'NDM', 'OXA beta-lactamase': 'OXA',
    'OXA-1-like beta-lactamase': 'OXA1', 'OXA-10-like beta-lactamase': 'OXA10',
    'OXA-134-like beta-lactamase': 'OXA134', 'OXA-2-like beta-lactamase': 'OXA2',
    'OXA-23-like beta-lactamase': 'OXA23', 'OXA-24-like beta-lactamase': 'OXA24',
    'OXA-46-like beta-lactamase': 'OXA46', 'OXA-48-like beta-lactamase': 'OXA48',
    'OXA-5-like beta-lactamase': 'OXA5', 'OXA-50-like beta-lactamase': 'OXA50',
    'OXA-51-like beta-lactamase': 'OXA51', 'OXA-58-like beta-lactamase': 'OXA58',
    'OXA-9-like beta-lactamase': 'OXA9', 'PDC beta-lactamase': 'PDC', 'PER beta-lactamase': 'PER',
    'PME beta-lactamase': 'PME',
    'Penicillin-binding protein mutations conferring resistance to beta-lactam antibiotics': 'PBP',
    'RSA beta-lactamase': 'RSA', 'SCO beta-lactamase': 'SCO', 'SED beta-lactamase': 'SED',
    'SFO beta-lactamase': 'SFO', 'SHV beta-lactamase': 'SHV', 'SIM beta-lactamase': 'SIM',
    'SPM beta-lactamase': 'SPM', 'TEM beta-lactamase': 'TEM', 'VEB beta-lactamase': 'VEB',
    'VIM beta-lactamase': 'VIM', 'ampC-type beta-lactamase': 'ampC',
}
shortcode_to_fullname = {v: k for k, v in all_beta_lactam_family_short_names.items()}

mapping = {}
with open(MAPPING_CSV) as f:
    for row in csv.DictReader(f):
        mapping[row['AMR_FAMILY_UNIQUE_NAMES']] = row['ID']   # already 'f9' etc, don't re-add the prefix

def sanitize(s):
    # whitespace + Newick syntax chars ( ) , : ; ' stripped -- 27% of headers in
    # this file are full free-text CARD descriptions and will corrupt a tree otherwise
    s = re.sub(r'\s+', '_', s.strip())
    return re.sub(r"[():,;']", '', s)

def read_fasta(path):
    records, header, seq = [], None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('>'):
                if header is not None:
                    records.append((header, seq))
                header, seq = line[1:].strip(), []
            else:
                seq.append(line)
        if header is not None:
            records.append((header, seq))
    return records

records = read_fasta(INPUT_FASTA)
lookup_rows = []

with open(OUTPUT_FASTA, 'w') as out:
    for old_header, seq in records:
        parts = old_header.split('_')
        idx, code, org = parts[-1], parts[-2], parts[-3]   # right-anchored: these 3 are closed
        gene_raw = '_'.join(parts[:-3])                     # known sets, safe even if gene_raw
        gene = sanitize(gene_raw)                            # itself contains underscores
        full_name = shortcode_to_fullname[code]
        fam_id = mapping[full_name]

        new_header = f'{fam_id}_{org}_{gene}_{idx}'
        out.write(f'>{new_header}\n')
        for line in seq:
            out.write(line + '\n')

        lookup_rows.append({
            'old_header': old_header, 'new_header': new_header,
            'organism': org, 'family_id': fam_id,
            'family_name': full_name, 'gene_allele': gene_raw,
        })

with open(OUTPUT_LOOKUP, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(lookup_rows[0].keys()))
    w.writeheader()
    w.writerows(lookup_rows)

assert len({r['new_header'] for r in lookup_rows}) == len(lookup_rows), 'collision -- stop and investigate'
print(f"{len(lookup_rows)} records relabelled -> {OUTPUT_FASTA}")
print(f"lookup table -> {OUTPUT_LOOKUP}")

## alignment of betalactam

In [135]:
from Bio.Seq import Seq

beta_lactam_relabelled_path = os.path.join(BETA_LACTAM_POOL_DIR, 'betalactam_cds_relabelled.fasta')

beta_lactam_frame_ok = []
beta_lactam_frame_bad = []
for header, seq in read_fasta_records(beta_lactam_relabelled_path):
    seq = seq.upper()
    if len(seq) % 3 != 0:
        beta_lactam_frame_bad.append((header, 'bad_length', len(seq)))
        continue
    protein = str(Seq(seq).translate(table=11, to_stop=False))
    body = protein[:-1] if protein.endswith('*') else protein
    if '*' in body:
        beta_lactam_frame_bad.append((header, 'internal_stop', body.index('*')))
        continue
    beta_lactam_frame_ok.append(header)

print(f'{len(beta_lactam_frame_ok)} / {len(beta_lactam_frame_ok) + len(beta_lactam_frame_bad)} sequences passed frame-integrity checks')
if beta_lactam_frame_bad:
    print('Failures:', beta_lactam_frame_bad[:10])

# Write only the passing sequences forward for alignment.
beta_lactam_relabelled_checked_path = os.path.join(BETA_LACTAM_POOL_DIR, 'betalactam_cds_relabelled_checked.fasta')
beta_lactam_frame_ok_set = set(beta_lactam_frame_ok)
with open(beta_lactam_relabelled_checked_path, 'w') as out:
    for header, seq in read_fasta_records(beta_lactam_relabelled_path):
        if header in beta_lactam_frame_ok_set:
            out.write(f'>{header}\n{seq}\n')

print(f'{len(beta_lactam_frame_ok)} sequences written to {beta_lactam_relabelled_checked_path} for alignment')

2934 / 2934 sequences passed frame-integrity checks
2934 sequences written to /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Phylogenetics/Beta_lactamase_all/betalactam_cds_relabelled_checked.fasta for alignment


In [136]:
import subprocess

beta_lactam_relabelled_aligned_path = os.path.join(BETA_LACTAM_POOL_DIR, 'betalactam_cds_relabelled_aligned.fasta')
mafft_relabelled_log_path = os.path.join(BETA_LACTAM_POOL_DIR, 'mafft_relabelled.log')

with open(beta_lactam_relabelled_aligned_path, 'w') as out, open(mafft_relabelled_log_path, 'w') as log:
    result = subprocess.run(
        [MAFFT_BIN, '--auto', '--thread', '16', '--inputorder', beta_lactam_relabelled_checked_path],
        stdout=out, stderr=log, text=True,
    )
if result.returncode != 0:
    raise RuntimeError(f'MAFFT failed, see {mafft_relabelled_log_path}')

beta_lactam_relabelled_aligned_records = read_fasta_records(beta_lactam_relabelled_aligned_path)
relabelled_alignment_lengths = {len(seq) for _, seq in beta_lactam_relabelled_aligned_records}
print(f'MAFFT alignment complete: {len(beta_lactam_relabelled_aligned_records)} sequences, alignment length {relabelled_alignment_lengths}')
print(f'-> {beta_lactam_relabelled_aligned_path}')

MAFFT alignment complete: 2934 sequences, alignment length {4726}
-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Phylogenetics/Beta_lactamase_all/betalactam_cds_relabelled_aligned.fasta


In [137]:
import re

# Verify headers are already Newick-safe -- sanitization happened at relabel
# time (before alignment), not after it like the old pipeline did. This
# replaces the old blind rewrite with a check: if it ever fails, that's a sign
# an unrelabelled or partially-sanitized file got fed in by mistake.
aligned_records = read_fasta_records(beta_lactam_relabelled_aligned_path)
unsafe_headers = [h for h, _ in aligned_records if re.search(r"[\s():,;']", h)]
if unsafe_headers:
    raise ValueError(
        f'{len(unsafe_headers)} headers still unsafe for Newick -- e.g. {unsafe_headers[0]!r}. '
        f'Something upstream of alignment was not fully relabelled/sanitized.'
    )
print(f'All {len(aligned_records)} headers verified Newick-safe -- no rewrite needed.')

# Build the tree from the relabelled alignment.
beta_lactam_relabelled_tree_path = os.path.join(BETA_LACTAM_POOL_DIR, 'cds_relabelled_tree.nwk')
fasttree_relabelled_log_path = os.path.join(BETA_LACTAM_POOL_DIR, 'fasttree_relabelled.log')

with open(beta_lactam_relabelled_tree_path, 'w') as out, open(fasttree_relabelled_log_path, 'w') as log:
    result = subprocess.run(
        [FASTTREE_BIN, '-nt', '-gtr', beta_lactam_relabelled_aligned_path],
        stdout=out, stderr=log, text=True,
    )
if result.returncode != 0:
    raise RuntimeError(f'FastTree failed, see {fasttree_relabelled_log_path}')

print(f'FastTree complete -> {beta_lactam_relabelled_tree_path}')

All 2934 headers verified Newick-safe -- no rewrite needed.
FastTree complete -> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Phylogenetics/Beta_lactamase_all/cds_relabelled_tree.nwk


## 25 Meta-data extraction 

## 25. Header relabel lookup for ALL gene families (not just beta-lactam)

Builds a lookup table for `All_Family_ID_Relabeled_2/CD/` -- the ID-relabeled, CD-HIT-deduplicated set covering every family in `AMR_Family_ID_Mapping.csv`, all six organisms -- in the same shape as the existing `header_relabel_lookup.csv` (beta-lactam only): `old_header, new_header, organism, family_id, family_name, gene_allele`.

Each new header (`f<ID>_<org>_<gene>_<i>`) already encodes its own gene name (spaces restored from underscores), so `gene_allele`/`old_header` could be decoded from the header alone -- but per the request, each decoded gene name is additionally cross-checked against the real header set in the corresponding organism's original `<org>_ARG_Fasta_Files/<family>.fasta` (the file as it existed before any relabeling), so the lookup is verified against the actual source data, not just trusted from the encoding.

**Per instruction: code only, not executed -- no files written.**

In [145]:
# CD-HIT output folder names are inconsistent (external tool artifact) -- confirmed by listing
# the actual directory: lowercase "pa", a double space before "Representative" for SA, etc.
ALL_FAMILY_CD_ROOT = os.path.join(GENEFAMILY_DIR, 'All_Family_ID_Relabeled_2', 'CD')
ALL_FAMILY_CD_ORG_FOLDERS = {
    'AB':  'AB Representative sequences',
    'EF':  'EF Representative sequences',
    'ENT': 'ENT Representative sequences',
    'Kp':  'Kp Representative sequences',
    'Pa':  'pa Representative sequences',
    'SA':  'SA  Representative sequences',
}

# Original (pre-relabel) per-organism folders -- the "previous names" source.
ORIGINAL_ORG_FOLDERS = {
    'AB':  'AB_ARG_Fasta_Files',
    'EF':  'EF_ARG_Fasta_Files',
    'ENT': 'ENT_ARG_Fasta_Files',
    'Kp':  'Kp_ARG_Fasta_Files',
    'Pa':  'Pa_ARG_Fasta_Files',
    'SA':  'SA_ARG_Fasta_Files',
}

# Reverse of family_name_to_id (built earlier from AMR_Family_ID_Mapping.csv) -- ID -> name.
family_id_to_name = {v: k for k, v in family_name_to_id.items()}

### 25.1 Decode each new header and validate against the original file

In [146]:
def parse_id_header(header):
    """Split an 'f<ID>_<org>_<gene>_<i>' header into (family_id, org, gene_allele)."""
    parts = header.split('_')
    family_id = parts[0]
    org = parts[1]
    gene_allele = '_'.join(parts[2:-1]).replace('_', ' ')
    return family_id, org, gene_allele


header_relabel_lookup_rows = []
mismatch_count = 0

for org, cd_folder in ALL_FAMILY_CD_ORG_FOLDERS.items():
    cd_dir = os.path.join(ALL_FAMILY_CD_ROOT, cd_folder)
    original_dir = os.path.join(GENEFAMILY_DIR, ORIGINAL_ORG_FOLDERS[org])

    for fasta_path in glob.glob(os.path.join(cd_dir, '*.fasta')):
        family_name = os.path.splitext(os.path.basename(fasta_path))[0]
        original_path = os.path.join(original_dir, f'{family_name}.fasta')

        original_headers = set()
        if os.path.exists(original_path):
            with open(original_path) as f:
                original_headers = {line[1:].strip() for line in f if line.startswith('>')}

        for new_header, seq in read_fasta_records(fasta_path):
            family_id, header_org, gene_allele = parse_id_header(new_header)
            family_name_from_id = family_id_to_name.get(family_id, '')

            if gene_allele not in original_headers:
                mismatch_count += 1

            header_relabel_lookup_rows.append({
                'old_header': gene_allele,
                'new_header': new_header,
                'organism': header_org,
                'family_id': family_id,
                'family_name': family_name_from_id,
                'gene_allele': gene_allele,
            })

all_family_header_relabel_lookup_df = pd.DataFrame(header_relabel_lookup_rows)
print(f'{len(all_family_header_relabel_lookup_df)} rows built')
print(f'{mismatch_count} gene_allele values NOT found in the original _ARG_Fasta_Files header set (expected: 0)')
all_family_header_relabel_lookup_df.head()

33965 rows built
0 gene_allele values NOT found in the original _ARG_Fasta_Files header set (expected: 0)


,old_header,new_header,organism,family_id,family_name,gene_allele
0,APH(4)-Ia,f20_AB_APH(4)-Ia_1,AB,f20,APH(4),APH(4)-Ia
1,adeN,f120_AB_adeN_1,AB,f120,resistance-nodulation-cell division (RND) anti...,adeN
2,adeS,f120_AB_adeS_2,AB,f120,resistance-nodulation-cell division (RND) anti...,adeS
3,adeR,f120_AB_adeR_3,AB,f120,resistance-nodulation-cell division (RND) anti...,adeR
4,adeB,f120_AB_adeB_5,AB,f120,resistance-nodulation-cell division (RND) anti...,adeB


### 25.2 Save

Not run per instruction -- shown for completeness so the table can be written once this section is executed.

In [147]:
all_family_header_lookup_path = os.path.join(
    GENEFAMILY_DIR, 'All_Family_ID_Relabeled_2', 'header_relabel_lookup_all_families.csv'
)
all_family_header_relabel_lookup_df.to_csv(all_family_header_lookup_path, index=False)
print(f'Saved -> {all_family_header_lookup_path}')

Saved -> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Genefamily/All_Family_ID_Relabeled_2/header_relabel_lookup_all_families.csv


## 26 Merge gene families of all the organism into a single .fasta 

## 26. Merge each gene family across organisms into one FASTA

For every distinct family filename found across the six `All_Family_ID_Relabeled_2/CD/` organism folders (section 25's `ALL_FAMILY_CD_ORG_FOLDERS`), pool whichever organisms actually have that family into one combined file of the same name -- e.g. `16S rRNA methyltransferase (G1405).fasta` exists for AB/ENT/Kp/Pa (not EF/SA); merging gives one file with all four organisms' sequences together.

No collision risk: headers are already `f<ID>_<org>_<gene>_<i>` (section 22's scheme), so the embedded `<org>` token alone guarantees no two organisms' records ever share a header, even before accounting for the per-file index.

Output goes to a new subfolder inside the same `CD` directory, as requested.

**Per instruction: code only, not executed -- no files written.**

In [148]:
ALL_FAMILY_MERGED_DIR = os.path.join(ALL_FAMILY_CD_ROOT, 'Merged_By_Family')
os.makedirs(ALL_FAMILY_MERGED_DIR, exist_ok=True)

# Discover every distinct family filename present across any organism's CD folder --
# not every organism has every family, so this is the union, not a fixed list.
all_family_filenames = set()
for org, folder in ALL_FAMILY_CD_ORG_FOLDERS.items():
    cd_dir = os.path.join(ALL_FAMILY_CD_ROOT, folder)
    for fasta_path in glob.glob(os.path.join(cd_dir, '*.fasta')):
        all_family_filenames.add(os.path.basename(fasta_path))

print(f'{len(all_family_filenames)} distinct family files found across all organisms')

142 distinct family files found across all organisms


### 26.1 Merge

In [149]:
merge_summary = {}
for filename in sorted(all_family_filenames):
    output_path = os.path.join(ALL_FAMILY_MERGED_DIR, filename)
    orgs_present = []
    total_records = 0

    with open(output_path, 'w') as out:
        for org, folder in ALL_FAMILY_CD_ORG_FOLDERS.items():
            input_path = os.path.join(ALL_FAMILY_CD_ROOT, folder, filename)
            if not os.path.exists(input_path):
                continue
            records = read_fasta_records(input_path)
            if not records:
                continue
            for header, seq in records:
                out.write(f'>{header}\n{seq}\n')
            orgs_present.append(org)
            total_records += len(records)

    merge_summary[filename] = {'organisms': orgs_present, 'n_organisms': len(orgs_present), 'n_records': total_records}

print(f'{len(merge_summary)} merged family files written to {ALL_FAMILY_MERGED_DIR}')

142 merged family files written to /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Genefamily/All_Family_ID_Relabeled_2/CD/Merged_By_Family


### 26.2 Verify

Header-uniqueness check per merged file (should always hold, since the `<org>` token embedded in every header makes cross-organism collisions structurally impossible), plus the notebook's own `16S rRNA methyltransferase (G1405)` example called out directly.

In [150]:
all_family_merge_summary_df = pd.DataFrame(
    [{'Family': k, **v} for k, v in merge_summary.items()]
).sort_values('n_records', ascending=False)

duplicate_header_files = []
for filename in sorted(all_family_filenames):
    output_path = os.path.join(ALL_FAMILY_MERGED_DIR, filename)
    headers = [h for h, s in read_fasta_records(output_path)]
    if len(headers) != len(set(headers)):
        duplicate_header_files.append(filename)

print(f'{len(duplicate_header_files)} merged files have duplicate headers (expected: 0)')
if duplicate_header_files:
    print(duplicate_header_files[:10])

example = '16S rRNA methyltransferase (G1405).fasta'
print()
print(f'Example -- {example}: {merge_summary[example]}')

all_family_merge_summary_df.head(10)

0 merged files have duplicate headers (expected: 0)

Example -- 16S rRNA methyltransferase (G1405).fasta: {'organisms': ['AB', 'ENT', 'Kp', 'Pa'], 'n_organisms': 4, 'n_records': 40}


,Family,organisms,n_organisms,n_records
120,resistance-nodulation-cell division (RND) anti...,"[AB, ENT, Kp, Pa]",4,14694
113,major facilitator superfamily (MFS) antibiotic...,"[AB, EF, ENT, Kp, Pa, SA]",6,3717
118,pmr phosphoethanolamine transferase.fasta,"[ENT, Kp, Pa]",3,2021
107,glycopeptide resistance gene cluster.fasta,"[EF, ENT, Kp, Pa, SA]",5,1021
21,ATP-binding cassette (ABC) antibiotic efflux p...,"[EF, ENT, Kp, Pa, SA]",5,865
117,nan.fasta,[Pa],1,820
67,Outer Membrane Porin (Opr).fasta,"[Kp, Pa]",2,694
101,elfamycin resistant EF-Tu.fasta,"[ENT, Kp]",2,661
123,small multidrug resistance (SMR) antibiotic ef...,"[AB, EF, ENT, Kp, Pa, SA]",6,569
38,General Bacterial Porin with reduced permeabil...,"[ENT, Kp]",2,524


## 27 Gene distribution metadata

## 27. Metadata for the merged (all-family, all-organism) gene distribution

Same three-file metadata pattern used throughout this notebook (see `Dedup_03_08_26/Gene_length_distribution.csv` / `_2.csv` / `_3.csv` for the original 30-family example), applied this time to section 26's merged output (`Merged_By_Family/`) -- every one of the 142 families, all six organisms.

One difference from the earlier sections: since each merged file now pools multiple organisms together, organism can no longer be read off which folder a file came from (section 17/20/23's approach) -- it has to be decoded from each individual header instead, reusing `parse_id_header` from section 25.

**Per instruction pattern established in sections 25-26: code only, not executed -- no files written.**

In [151]:
def parse_merged_family_fasta(path):
    """
    Parse one merged (all-organisms-pooled) family FASTA, decoding organism, family ID/name,
    gene name, and sequence length directly from each 'f<ID>_<org>_<gene>_<i>' header --
    reuses parse_id_header (section 25) for the header decode.
    Returns a list of (organism, family_id, family_name, gene_allele, length) tuples.
    """
    rows = []
    for header, seq in read_fasta_records(path):
        family_id, org, gene_allele = parse_id_header(header)
        family_name = family_id_to_name.get(family_id, '')
        rows.append((org, family_id, family_name, gene_allele, len(seq)))
    return rows

### 27.1 Build the long-format table (all merged families, all organisms)

In [152]:
merged_gene_length_rows = []
for filename in sorted(all_family_filenames):
    path = os.path.join(ALL_FAMILY_MERGED_DIR, filename)
    if not os.path.exists(path):
        continue
    for org, family_id, family_name, gene_allele, length in parse_merged_family_fasta(path):
        merged_gene_length_rows.append({
            'Organism': org,
            'AMR Gene Family': family_name,
            'Gene': gene_allele,
            'Length': length,
        })

merged_gene_length_long_df = pd.DataFrame(merged_gene_length_rows)
print(f'{len(merged_gene_length_long_df)} total records across all merged family files')
merged_gene_length_long_df.head()

33965 total records across all merged family files


,Organism,AMR Gene Family,Gene,Length
0,AB,16S rRNA methyltransferase (G1405),armA,816
1,AB,16S rRNA methyltransferase (G1405),armA,774
2,AB,16S rRNA methyltransferase (G1405),armA,690
3,AB,16S rRNA methyltransferase (G1405),armA,774
4,AB,16S rRNA methyltransferase (G1405),armA,774


### 27.2 Gene_length_distribution.csv -- per (Organism, Family, Gene) summary

Reuses `gene_length_per_family_per_organism` (section 16) unchanged.

In [153]:
merged_gene_length_per_organism_df = gene_length_per_family_per_organism(merged_gene_length_long_df)

gene_length_distribution_path = os.path.join(ALL_FAMILY_MERGED_DIR, 'Gene_length_distribution.csv')
merged_gene_length_per_organism_df.to_csv(gene_length_distribution_path)
print(f'{len(merged_gene_length_per_organism_df)} rows -> {gene_length_distribution_path}')
merged_gene_length_per_organism_df.head()

1897 rows -> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Genefamily/All_Family_ID_Relabeled_2/CD/Merged_By_Family/Gene_length_distribution.csv


,Organism,AMR Gene Family,Gene,N_Records,Min_Length,Max_Length,N_Distinct_Lengths
0,AB,16S rRNA methyltransferase (G1405),armA,5,690,816,3
460,ENT,16S rRNA methyltransferase (G1405),armA,2,774,774,1
815,Kp,16S rRNA methyltransferase (G1405),armA,8,774,774,1
1285,Pa,16S rRNA methyltransferase (G1405),armA,1,774,774,1
461,ENT,16S rRNA methyltransferase (G1405),rmtB,3,756,756,1


### 27.3 Gene_length_distribution_2.csv -- per-record long format

In [154]:
gene_length_distribution_2_path = os.path.join(ALL_FAMILY_MERGED_DIR, 'Gene_length_distribution_2.csv')
merged_gene_length_long_df.to_csv(gene_length_distribution_2_path)
print(f'{len(merged_gene_length_long_df)} rows -> {gene_length_distribution_2_path}')

33965 rows -> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Genefamily/All_Family_ID_Relabeled_2/CD/Merged_By_Family/Gene_length_distribution_2.csv


### 27.4 Gene_length_distribution_3.csv -- per (Organism, Family) summary, no gene breakdown

In [155]:
merged_organism_family_length_summary_df = (
    merged_gene_length_long_df
    .groupby(['Organism', 'AMR Gene Family'])
    .agg(
        N_Records=('Length', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
    )
    .reset_index()
    .sort_values(['AMR Gene Family', 'Organism'])
)

gene_length_distribution_3_path = os.path.join(ALL_FAMILY_MERGED_DIR, 'Gene_length_distribution_3.csv')
merged_organism_family_length_summary_df.to_csv(gene_length_distribution_3_path)
print(f'{len(merged_organism_family_length_summary_df)} rows -> {gene_length_distribution_3_path}')
merged_organism_family_length_summary_df.head()

345 rows -> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Genefamily/All_Family_ID_Relabeled_2/CD/Merged_By_Family/Gene_length_distribution_3.csv


,Organism,AMR Gene Family,N_Records,Min_Length,Max_Length
0,AB,16S rRNA methyltransferase (G1405),6,690,846
82,ENT,16S rRNA methyltransferase (G1405),6,756,846
155,Kp,16S rRNA methyltransferase (G1405),21,690,858
232,Pa,16S rRNA methyltransferase (G1405),7,744,780
295,SA,23S rRNA with mutation conferring resistance t...,4,2926,2926


In [157]:
import pandas as pd

# 1. Load the CSV file into a pandas DataFrame
# Replace 'your_data.csv' with the actual path to your file
df = pd.read_csv('/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Genefamily/All_Family_ID_Relabeled_2/CD/Gene_length_distribution_3.csv')

# 2. Define your threshold limit
threshold_value = 8.0  # Change this to whatever cutoff number you need

# 3. Filter the DataFrame and select Column A
# How this reads: "Go into 'df', find where 'Column_B' > threshold, and give me 'Column_A'"
filtered_names = df.loc[df['N_Records'] > threshold_value, 'AMR Gene Family']

# 4. Convert the result to a clean list and display it
names_list = filtered_names.tolist()

print(f"Found {len(names_list)} cells where Column B is greater than {threshold_value}:")
print(names_list)

Found 165 cells where Column B is greater than 8.0:
['16S rRNA methyltransferase (G1405)', 'AAC(3)', 'AAC(3)', "AAC(6')", "AAC(6')", "AAC(6')", "AAC(6')", "AAC(6')", 'ACT beta-lactamase', 'ADC beta-lactamase without carbapenemase activity', 'ADC beta-lactamases pending classification for carbapenemase activity', "ANT(2'')", "ANT(3'')", "ANT(3'')", "ANT(3'')", "ANT(3'')", 'ANT(6)', 'ANT(9)', "APH(3'')", "APH(3'')", "APH(3'')", "APH(3')", "APH(3')", "APH(3')", "APH(3')", "APH(3')", 'APH(6)', 'APH(6)', 'ATP-binding cassette (ABC) antibiotic efflux pump', 'ATP-binding cassette (ABC) antibiotic efflux pump', 'ATP-binding cassette (ABC) antibiotic efflux pump', 'ATP-binding cassette (ABC) antibiotic efflux pump', 'ATP-binding cassette (ABC) antibiotic efflux pump', 'BlaZ beta-lactamase', 'CMH beta-lactamase', 'CMY beta-lactamase', 'CTX-M beta-lactamase', 'CTX-M beta-lactamase', 'Erm 23S ribosomal RNA methyltransferase', 'Erm 23S ribosomal RNA methyltransferase', 'GES beta-lactamase', 'Genera

In [163]:
import pandas as pd

# 1. Load the CSV file into a pandas DataFrame
# Replace 'your_data.csv' with the actual path to your file
df = pd.read_csv('/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Genefamily/All_Family_ID_Relabeled_2/CD/Gene_length_distribution_3.csv')

# 2. Define your threshold limit
threshold_value = 20.0  # Change this to your specific cutoff number

# 3. Filter the DataFrame (keep all columns for now, just drop the rows that fail the test)
filtered_df = df[df['N_Records'] > threshold_value]

# 4. Group by Column C, and aggregate the passing names from Column A into a list
grouped_df = filtered_df.groupby('Organism')['AMR Gene Family'].apply(list).reset_index()

# 5. Rename the aggregated column for clarity (optional but recommended)
grouped_df.rename(columns={'AMR Gene Family': 'Passing_Names'}, inplace=True)

# 6. Display the final grouped DataFrame
# If running in Jupyter Notebook, just calling the variable or using display() looks best
display(grouped_df)

,Organism,Passing_Names
0,AB,[ADC beta-lactamase without carbapenemase acti...
1,EF,"[Erm 23S ribosomal RNA methyltransferase, Misc..."
2,ENT,"[AAC(6'), ACT beta-lactamase, ANT(3''), ATP-bi..."
3,Kp,"[16S rRNA methyltransferase (G1405), AAC(3), A..."
4,Pa,"[AAC(6'), ANT(3''), APH(3'), ATP-binding casse..."
5,SA,[ATP-binding cassette (ABC) antibiotic efflux ...


In [160]:
grouped_df.head(15).style

,Organism,Passing_Names
0,AB,"['AAC(3)', ""AAC(6')"", 'ADC beta-lactamase without carbapenemase activity', 'ADC beta-lactamases pending classification for carbapenemase activity', ""ANT(2'')"", ""ANT(3'')"", ""APH(3'')"", ""APH(3')"", 'APH(6)', 'Intrinsic peptide antibiotic resistant Lps', 'OXA beta-lactamase', 'OXA-23-like beta-lactamase', 'OXA-51-like beta-lactamase', 'fluoroquinolone resistant gyrA', 'fluoroquinolone resistant parC', 'major facilitator superfamily (MFS) antibiotic efflux pump', 'multidrug and toxic compound extrusion (MATE) transporter', 'resistance-nodulation-cell division (RND) antibiotic efflux pump', 'small multidrug resistance (SMR) antibiotic efflux pump', 'sulfonamide resistant sul']"
1,EF,"[""AAC(6')"", 'ANT(6)', 'ATP-binding cassette (ABC) antibiotic efflux pump', 'Erm 23S ribosomal RNA methyltransferase', 'Miscellaneous ABC-F subfamily ATP-binding cassette ribosomal protection proteins', 'Van ligase', 'aminoglycoside bifunctional resistance protein', 'glycopeptide resistance gene cluster', 'major facilitator superfamily (MFS) antibiotic efflux pump', 'tetracycline-resistant ribosomal protection protein', 'vanH', 'vanR', 'vanS', 'vanW', 'vanX', 'vanY']"
2,ENT,"[""AAC(6')"", 'ACT beta-lactamase', ""ANT(3'')"", ""APH(3'')"", ""APH(3')"", 'ATP-binding cassette (ABC) antibiotic efflux pump', 'CMH beta-lactamase', 'CTX-M beta-lactamase', 'General Bacterial Porin with reduced permeability to beta-lactams', 'IMI beta-lactamase', 'IMP beta-lactamase', 'MCR phosphoethanolamine transferase', 'MIR beta-lactamase', 'Penicillin-binding protein mutations conferring resistance to beta-lactam antibiotics', 'Van ligase', 'antibiotic-resistant UhpT', 'elfamycin resistant EF-Tu', 'fluoroquinolone resistant gyrA', 'fluoroquinolone resistant parC', 'fosfomycin thiol transferase', 'glycopeptide resistance gene cluster', 'macrolide phosphotransferase (MPH)', 'major facilitator superfamily (MFS) antibiotic efflux pump', 'pmr phosphoethanolamine transferase', 'quinolone resistance protein (qnr)', 'resistance-nodulation-cell division (RND) antibiotic efflux pump', 'small multidrug resistance (SMR) antibiotic efflux pump', 'sulfonamide resistant sul', 'trimethoprim resistant dihydrofolate reductase dfr', 'undecaprenyl pyrophosphate related proteins']"
3,Kp,"['16S rRNA methyltransferase (G1405)', 'AAC(3)', ""AAC(6')"", ""ANT(3'')"", ""APH(3'')"", ""APH(3')"", 'APH(6)', 'ATP-binding cassette (ABC) antibiotic efflux pump', 'CMY beta-lactamase', 'CTX-M beta-lactamase', 'General Bacterial Porin with reduced permeability to beta-lactams', 'General Bacterial Porin with reduced permeability to peptide antibiotics', 'KPC beta-lactamase', 'MCR phosphoethanolamine transferase', 'NDM beta-lactamase', 'OXA beta-lactamase', 'OXA-1-like beta-lactamase', 'OXA-48-like beta-lactamase', 'Outer Membrane Porin (Opr)', 'Penicillin-binding protein mutations conferring resistance to beta-lactam antibiotics', 'SHV beta-lactamase', 'TEM beta-lactamase', 'Van ligase', 'antibiotic-resistant UhpT', 'chloramphenicol acetyltransferase (CAT)', 'elfamycin resistant EF-Tu', 'fluoroquinolone resistant gyrA', 'fluoroquinolone resistant gyrB', 'fluoroquinolone resistant parC', 'fosfomycin thiol transferase', 'glycopeptide resistance gene cluster', 'macrolide phosphotransferase (MPH)', 'major facilitator superfamily (MFS) antibiotic efflux pump', 'multidrug and toxic compound extrusion (MATE) transporter', 'pmr phosphoethanolamine transferase', 'quinolone resistance protein (qnr)', 'resistance-nodulation-cell division (RND) antibiotic efflux pump', 'rifampin ADP-ribosyltransferase (Arr)', 'rifamycin-resistant beta-subunit of RNA polymerase (rpoB)', 'small multidrug resistance (SMR) antibiotic efflux pump', 'sulfonamide resistant sul', 'triclosan resistant gyrA', 'trimethoprim resistant dihydrofolate reductase dfr']"
4,Pa,"[""AAC(6')"", ""ANT(3'')"", ""APH(3')"", 'ATP-binding cassette (ABC) antibiotic efflux pump', 'GES beta-lactamase', 'IMP beta-lactamase', 'KPC

In [162]:
import os

# Removed the accidental double-paste and closed the to_csv() function properly
grouped_df.to_csv(
    os.path.join('/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/Genefamily/All_Family_ID_Relabeled_2/CD/families.csv'), 
    index=False
)

print("File successfully saved!")

File successfully saved!
